# Integrated Colombian 16S pipeline

This is the clean head-to-tail workflow assembled from the saved Week 1 and ML notebooks. Change only the repository's `config/preferences.yaml`. Set `run.branch` to `asv` or `otu`; the notebook passes that branch directly to Nextflow.

The OTU path uses DADA2 as its prerequisite denoising step, then clusters representative sequences at the configured similarity. It does not run ASV and OTU downstream analyses together.

The two switches are `biology.enabled` for DADA2/ASV/OTU/diversity and `ml.enabled` for machine learning/ML plots. Set either or both to `true`; no separate mode parameter is needed.


In [ ]:
%pip -q install pyyaml openpyxl scikit-learn biom-format


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 107.2 MB/s eta 0:00:00


In [ ]:
from pathlib import Path
import hashlib, os, shutil, subprocess, tarfile
import pandas as pd
import yaml
from google.colab import drive

drive.mount('/content/drive')
INTEGRATION_REPOSITORY = 'https://github.com/bcnbsr/integrated-16s-microbiome-ml.git'
PROJECT_ROOT = Path('/content/project')
REPO_DIR = PROJECT_ROOT / 'integrated-16s-microbiome-ml'
if REPO_DIR.exists() and not (REPO_DIR / '.git').is_dir():
    raise RuntimeError(f'Refusing to overwrite non-repository directory: {REPO_DIR}')
if not (REPO_DIR / '.git').is_dir():
    PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
    subprocess.run(['git', 'clone', INTEGRATION_REPOSITORY, str(REPO_DIR)], check=True)
PREFERENCES_FILE = REPO_DIR / 'config/preferences.yaml'
if not PREFERENCES_FILE.is_file():
    raise FileNotFoundError(f'Active repository configuration is missing: {PREFERENCES_FILE}')
CONFIG = yaml.safe_load(PREFERENCES_FILE.read_text())
if CONFIG['input']['repository'].rstrip('/') != INTEGRATION_REPOSITORY.rstrip('/'):
    raise ValueError('config/preferences.yaml must identify this integration repository.')

# Two independent switches in config/preferences.yaml:
#   biology.enabled -> DADA2, ASV/OTU processing, and diversity plots
#   ml.enabled      -> machine-learning analysis and ML plots
BIOLOGY_ENABLED = bool(CONFIG.get('biology', {}).get('enabled', True))
ML_ENABLED = bool(CONFIG.get('ml', {}).get('enabled', True))
if not BIOLOGY_ENABLED and not ML_ENABLED:
    raise ValueError("Enable at least one pipeline part: biology.enabled or ml.enabled.")
os.environ['BIOLOGY_ENABLED'] = str(BIOLOGY_ENABLED).lower()
os.environ['ML_ENABLED'] = str(ML_ENABLED).lower()

BRANCH = CONFIG['run']['branch']
if BRANCH not in {'asv', 'otu'}:
    raise ValueError("run.branch must be 'asv' or 'otu'")
BRANCH_KEY = BRANCH
NEXTFLOW_BRANCH = BRANCH

if Path(CONFIG['run']['project_dir']) != PROJECT_ROOT:
    raise ValueError('run.project_dir must be /content/project for this notebook.')
RESULTS_ROOT = Path(CONFIG['run']['results_dir'])
ML_ROOT = Path(CONFIG['run']['ml_dir'])
if Path(CONFIG['input']['repository_dir']) != REPO_DIR:
    raise ValueError('input.repository_dir must match the cloned integration repository.')
WORKFLOW_DIR = REPO_DIR / CONFIG['input']['workflow_dir']
if not (WORKFLOW_DIR / 'main.nf').is_file():
    raise FileNotFoundError(f'Nextflow workflow is missing: {WORKFLOW_DIR / "main.nf"}')
ml_implementation_dir = Path(CONFIG['ml']['implementation_dir'])
if ml_implementation_dir.is_absolute():
    raise ValueError('ml.implementation_dir must be relative to the integration repository.')
ML_REPO = (REPO_DIR / ml_implementation_dir).resolve()
if REPO_DIR.resolve() not in ML_REPO.parents:
    raise ValueError('ml.implementation_dir must stay within the integration repository.')
MODEL_ID_COLUMN = CONFIG['metadata']['phenotype_id_column']
OUTCOME_COLUMN = CONFIG['metadata']['phenotype_group_column']
DIET_COLUMNS = tuple(CONFIG['ml']['diet_columns'])
PHENOTYPE_FILE = ML_ROOT / Path(CONFIG['input']['phenotype_metadata']).name
RUN_MAPPING_FILE = ML_ROOT / Path(CONFIG['input']['run_participant_map']).name
DRIVE_ROOT = Path(CONFIG['input']['drive_root'])
QIIME_BIN = Path('/content/miniforge/envs/qiime2-amplicon-2024.10/bin/qiime')
BIOM_BIN = Path('/content/miniforge/envs/qiime2-amplicon-2024.10/bin/biom')
os.environ['ARCHIVE_DIR'] = CONFIG['run']['archive_dir']
os.environ['RUN_NAME'] = CONFIG['run']['name']
os.environ['BRANCH_KEY'] = BRANCH_KEY
os.environ['QIIME_BIN'] = str(QIIME_BIN)
os.environ['REPO_DIR'] = str(REPO_DIR)
os.environ['READS_DIR'] = CONFIG['reads']['directory']

for key in ['trunc_len_f', 'trunc_len_r', 'sampling_depth', 'min_quality', 'max_expected_errors']:
    if not isinstance(CONFIG['quality'][key], int) or CONFIG['quality'][key] < 0:
        raise ValueError(f'Invalid quality.{key}')
if not 0 < float(CONFIG['quality']['error_rate']) <= 1:
    raise ValueError('Invalid quality.error_rate')

print('Configured branch:', BRANCH)
print('Biology enabled:', BIOLOGY_ENABLED, '| ML enabled:', ML_ENABLED)
print('Parameters:', CONFIG['quality'])


Mounted at /content/drive


Saving preferences_colombian.yaml to preferences_colombian.yaml
Configured branch: otu
Biology enabled: True | ML enabled: True
Parameters: {'trunc_len_f': 283, 'trunc_len_r': 229, 'sampling_depth': 27892, 'min_quality': 28, 'max_expected_errors': 2, 'error_rate': 0.1, 'min_length': 0, 'trim_left_f': 0, 'trim_left_r': 0}


## 1. Environment, inputs, and reusable helpers

Run this section in order. Reusable helpers are defined here but are executed only in their labelled downstream sections.


In [ ]:
%%bash
set -euo pipefail

mkdir -p /content/project/{repo,data,results,refs,logs,tmp}
mkdir -p /content/nxf_work

mkdir -p /content/drive/MyDrive/microbiome_colab/{repo_backup,results,refs,logs}

echo "Project folders ready."
df -h /content


Project folders ready.
Filesystem      Size  Used Avail Use% Mounted on
overlay         226G   21G  206G   9% /


In [ ]:
# Reusable diversity-plot helper. It is intentionally defined during setup
# and invoked only after Nextflow has completed the selected branch in section 3.

def plot_diversity_outputs():
    if BIOLOGY_ENABLED:
        import matplotlib.pyplot as plt
        import numpy as np
        import pandas as pd

        DIVERSITY_PLOT_DIR = BRANCH_DIR / "diversity" / "plots"
        DIVERSITY_EXPORT_DIR = BRANCH_DIR / "diversity" / "exports"
        DIVERSITY_PLOT_DIR.mkdir(parents=True, exist_ok=True)
        DIVERSITY_EXPORT_DIR.mkdir(parents=True, exist_ok=True)

        diversity_metadata = pd.read_csv(QIIME_METADATA, sep="\t", dtype=str).fillna("")
        diversity_group = "group" if "group" in diversity_metadata.columns else "dep_flag_corrected"
        if diversity_group not in diversity_metadata.columns:
            raise ValueError("No group column is available for diversity plots.")
        if diversity_group == "dep_flag_corrected":
            diversity_metadata[diversity_group] = diversity_metadata[diversity_group].map({"0": "healthy", "1": "depressive"}).fillna(diversity_metadata[diversity_group])
        diversity_metadata = diversity_metadata[["sample-id", diversity_group]]
        diversity_metadata["sample-id"] = diversity_metadata["sample-id"].astype(str).str.strip()

        def export_qza(artifact, name):
            artifact = Path(artifact)
            if not artifact.is_file():
                return None
            target = DIVERSITY_EXPORT_DIR / name
            if target.exists():
                shutil.rmtree(target)
            target.mkdir(parents=True)
            qrun("tools", "export", "--input-path", artifact, "--output-path", target)
            return target

        def exported_file(directory, name):
            candidate = Path(directory) / name
            if candidate.is_file():
                return candidate
            files = sorted(Path(directory).glob("*"))
            return files[0] if files else None

        alpha_sources = {
            "Shannon": DIVERSITY_DIR / "shannon_vector.qza",
            "Observed features": DIVERSITY_DIR / "observed_features_vector.qza",
            "Pielou evenness": DIVERSITY_DIR / "evenness_vector.qza",
            "Faith's PD": DIVERSITY_DIR / "faith_pd_vector.qza",
            "Chao1": CHAO1,
        }
        alpha_frames = []
        for metric, artifact in alpha_sources.items():
            directory = export_qza(artifact, "alpha_" + metric.lower().replace(" ", "_").replace("'", ""))
            if directory is None:
                print("Skipping unavailable alpha metric:", metric)
                continue
            table = exported_file(directory, "alpha-diversity.tsv")
            if table is None:
                continue
            values = pd.read_csv(table, sep="\t")
            sample_column, value_column = values.columns[0], values.columns[1]
            values = values.rename(columns={sample_column: "sample-id", value_column: "value"})
            values["sample-id"] = values["sample-id"].astype(str).str.strip()
            values["value"] = pd.to_numeric(values["value"], errors="coerce")
            values["Metric"] = metric
            alpha_frames.append(values[["sample-id", "Metric", "value"]])

        if alpha_frames:
            alpha = pd.concat(alpha_frames, ignore_index=True).merge(diversity_metadata, on="sample-id", how="inner")
            alpha.to_csv(DIVERSITY_PLOT_DIR / "alpha_diversity_metrics.tsv", sep="\t", index=False)
            metrics = [metric for metric in alpha_sources if metric in set(alpha["Metric"])]
            groups = sorted(alpha[diversity_group].dropna().unique())
            fig, axes = plt.subplots(1, len(metrics), figsize=(4 * len(metrics), 4.8), squeeze=False)
            axes = axes.ravel(); rng = np.random.default_rng(42)
            for axis, metric in zip(axes, metrics):
                subset = alpha[alpha["Metric"] == metric]
                data = [subset.loc[subset[diversity_group] == group, "value"].dropna().to_numpy() for group in groups]
                axis.boxplot(data, patch_artist=True, boxprops={"facecolor": "#D9E2F3"}, medianprops={"color": "black"})
                for position, values in enumerate(data, 1):
                    if len(values):
                        axis.scatter(position + rng.normal(0, .05, len(values)), values, s=18, alpha=.75)
                axis.set_title(metric, fontweight="bold"); axis.set_xticks(range(1, len(groups) + 1)); axis.set_xticklabels(groups, rotation=35, ha="right"); axis.grid(axis="y", alpha=.25)
            fig.suptitle("Alpha diversity — " + BRANCH_KEY, fontweight="bold"); fig.tight_layout()
            fig.savefig(DIVERSITY_PLOT_DIR / "alpha_diversity_metrics.png", dpi=300, bbox_inches="tight")
            fig.savefig(DIVERSITY_PLOT_DIR / "alpha_diversity_metrics.pdf", bbox_inches="tight"); plt.close(fig)
        else:
            print("No alpha-diversity vectors were available for plotting.")

        beta_sources = {
            "Jaccard": DIVERSITY_DIR / "jaccard_distance.qza",
            "Bray-Curtis": DIVERSITY_DIR / "bray_curtis_distance.qza",
            "Unweighted UniFrac": DIVERSITY_DIR / "unweighted_unifrac_distance.qza",
            "Weighted UniFrac": DIVERSITY_DIR / "weighted_unifrac_distance.qza",
        }
        def read_distance(path):
            lines = Path(path).read_text().splitlines(); header = next(i for i, line in enumerate(lines) if line.startswith("#SampleID") or line.startswith("sample-id"))
            table = pd.read_csv(path, sep="\t", skiprows=header); first = table.columns[0]; table[first] = table[first].astype(str).str.strip()
            return table.set_index(first).apply(pd.to_numeric, errors="coerce")
        def pcoa(distance):
            matrix = distance.to_numpy(float); n = matrix.shape[0]; center = np.eye(n) - np.ones((n, n)) / n; gram = -.5 * center @ (matrix ** 2) @ center
            eigenvalues, eigenvectors = np.linalg.eigh(gram); order = np.argsort(eigenvalues)[::-1]; eigenvalues = eigenvalues[order]; eigenvectors = eigenvectors[:, order]; positive = eigenvalues > 1e-12
            eigenvalues, eigenvectors = eigenvalues[positive], eigenvectors[:, positive]
            if len(eigenvalues) < 2: raise ValueError("The distance matrix has fewer than two positive PCoA axes.")
            return eigenvectors[:, :2] * np.sqrt(eigenvalues[:2]), eigenvalues[:2] / eigenvalues.sum()

        beta_panels = []
        known_samples = set(diversity_metadata["sample-id"])
        for metric, artifact in beta_sources.items():
            directory = export_qza(artifact, "beta_" + metric.lower().replace(" ", "_").replace("-", ""))
            if directory is None:
                print("Skipping unavailable beta metric:", metric); continue
            table = exported_file(directory, "distance-matrix.tsv")
            if table is None: continue
            distance = read_distance(table); ids = [sample for sample in distance.index if sample in known_samples]
            if len(ids) < 3:
                print("Skipping beta metric with fewer than three matched samples:", metric)
                continue
            distance = distance.loc[ids, ids]
            coordinates, explained = pcoa(distance)
            frame = pd.DataFrame(coordinates, index=distance.index, columns=["PCoA1", "PCoA2"]).reset_index(names="sample-id").merge(diversity_metadata, on="sample-id", how="left")
            frame.to_csv(DIVERSITY_PLOT_DIR / ("beta_" + metric.lower().replace(" ", "_").replace("-", "") + "_pcoa.tsv"), sep="\t", index=False)
            beta_panels.append((metric, explained, frame))

        if beta_panels:
            fig, axes = plt.subplots(2, 2, figsize=(10, 8), squeeze=False); axes = axes.ravel()
            for axis, (metric, explained, frame) in zip(axes, beta_panels):
                for group in sorted(frame[diversity_group].dropna().unique()):
                    part = frame[frame[diversity_group] == group]; axis.scatter(part.PCoA1, part.PCoA2, s=32, alpha=.8, label=group)
                axis.set_title(metric, fontweight="bold"); axis.set_xlabel(f"PCoA1 ({explained[0] * 100:.1f}%)"); axis.set_ylabel(f"PCoA2 ({explained[1] * 100:.1f}%)"); axis.grid(alpha=.25); axis.legend(frameon=False, fontsize=8)
            for axis in axes[len(beta_panels):]: axis.set_visible(False)
            fig.suptitle("Beta diversity PCoA — " + BRANCH_KEY, fontweight="bold"); fig.tight_layout()
            fig.savefig(DIVERSITY_PLOT_DIR / "beta_diversity_pcoa.png", dpi=300, bbox_inches="tight")
            fig.savefig(DIVERSITY_PLOT_DIR / "beta_diversity_pcoa.pdf", bbox_inches="tight"); plt.close(fig)
        else:
            print("No beta-diversity distance matrices were available for plotting.")
        print("Diversity plots:"); [print(path) for path in sorted(DIVERSITY_PLOT_DIR.iterdir())]
    else:
        print("Biology disabled: skipping ASV/OTU diversity plots.")


In [ ]:
%%bash
set -euo pipefail
if [ "${BIOLOGY_ENABLED:-true}" = "false" ]; then
  echo "Biology disabled: skipping system and Nextflow setup."
  exit 0
fi

apt-get update -qq
apt-get install -y -qq openjdk-17-jdk wget curl unzip pigz git

java -version

if ! command -v nextflow >/dev/null 2>&1; then
  curl -s https://get.nextflow.io | bash
  chmod +x nextflow
  mv nextflow /usr/local/bin/
fi

nextflow -version


(Reading database ... 118422 files and directories currently installed.)
Preparing to unpack .../00-wget_1.21.2-2ubuntu1.5_amd64.deb ...
Unpacking wget (1.21.2-2ubuntu1.5) over (1.21.2-2ubuntu1.4) ...
Selecting previously unselected package libatspi2.0-0:amd64.
Preparing to unpack .../01-libatspi2.0-0_2.44.0-3_amd64.deb ...
Unpacking libatspi2.0-0:amd64 (2.44.0-3) ...
Selecting previously unselected package libxtst6:amd64.
Preparing to unpack .../02-libxtst6_2%3a1.2.3-1build4_amd64.deb ...
Unpacking libxtst6:amd64 (2:1.2.3-1build4) ...
Selecting previously unselected package session-migration.
Preparing to unpack .../03-session-migration_0.3.6_amd64.deb ...
Unpacking session-migration (0.3.6) ...
Selecting previously unselected package gsettings-desktop-schemas.
Preparing to unpack .../04-gsettings-desktop-schemas_42.0-1ubuntu1_all.deb ...
Unpacking gsettings-desktop-schemas (42.0-1ubuntu1) ...
Selecting previously unselected package at-spi2-core.
Preparing to unpack .../05-at-spi2-cor

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
openjdk version "21.0.11" 2026-04-21
OpenJDK Runtime Environment (build 21.0.11+10-1-22.04.2-Ubuntu)
OpenJDK 64-Bit Server VM (build 21.0.11+10-1-22.04.2-Ubuntu, mixed mode, sharing)


In [ ]:
%%bash
set -euo pipefail
if [ "${BIOLOGY_ENABLED:-true}" = "false" ]; then
  echo "Biology disabled: skipping Miniforge setup."
  exit 0
fi

MINIFORGE=/content/miniforge

if [ ! -d "$MINIFORGE" ]; then
  wget -qO /content/miniforge.sh \
    https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh

  bash /content/miniforge.sh -b -p "$MINIFORGE"
fi

source "$MINIFORGE/etc/profile.d/conda.sh"

conda config --set channel_priority strict
conda --version

echo "Conda ready."


conda 26.5.3
Conda ready.


In [ ]:
%%bash
set -euo pipefail
if [ "${BIOLOGY_ENABLED:-true}" = "false" ]; then
  echo "Biology disabled: skipping QIIME2 environment setup."
  exit 0
fi

source /content/miniforge/etc/profile.d/conda.sh

cd /content/project

echo "Removing any partial failed QIIME2 environment..."
conda env remove -n qiime2-amplicon-2024.10 -y || true

echo "Cleaning conda cache..."
conda clean --all -y

echo "Setting channel priority to flexible. This is required for the deblur/sortmerna issue."
conda config --set channel_priority flexible

echo "Current conda config:"
conda config --show channel_priority

echo "Downloading QIIME2 2024.10 amplicon environment file..."
rm -f qiime2-amplicon-2024.10-py310-linux-conda.yml

wget -O qiime2-amplicon-2024.10-py310-linux-conda.yml \
  https://data.qiime2.org/distro/amplicon/qiime2-amplicon-2024.10-py310-linux-conda.yml

echo "Creating QIIME2 environment. This may take a while..."
conda env create \
  -n qiime2-amplicon-2024.10 \
  --file qiime2-amplicon-2024.10-py310-linux-conda.yml

echo "Activating QIIME2..."
conda activate qiime2-amplicon-2024.10

echo "Testing QIIME2..."
qiime info
qiime --help | head -n 20


Removing any partial failed QIIME2 environment...
Cleaning conda cache...
Will remove 123 (101.3 MB) tarball(s).
Will remove 1 index cache(s).
Will remove 11 (5.6 MB) package(s).
There are no tempfile(s) to remove.
There are no logfile(s) to remove.
Setting channel priority to flexible. This is required for the deblur/sortmerna issue.
Current conda config:
channel_priority: flexible
Creating QIIME2 environment. This may take a while...
Retrieving notices: - \ done
Channels:
 - https://packages.qiime2.org/qiime2/2024.10/amplicon/released
 - conda-forge
 - bioconda
Platform: linux-64
Solving environment: - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / - \ done

openjdk-22.0.1       | 173.2 MB  |            |   0% 
blast-2.16.0         | 141.1 MB  |            |   0% 

gcc_impl_linux-64-14 | 70.1 MB   |            |   0% 


qt-main-5.15.8       | 58.5 MB   |            |   0% 



pillow-10.3.0        | 39.8 MB   |    


EnvironmentLocationNotFound: Not a conda environment: /content/miniforge/envs/qiime2-amplicon-2024.10

--2026-08-25 11:57:45--  https://data.qiime2.org/distro/amplicon/qiime2-amplicon-2024.10-py310-linux-conda.yml
Resolving data.qiime2.org (data.qiime2.org)... 54.200.1.12
Connecting to data.qiime2.org (data.qiime2.org)|54.200.1.12|:443... connected.
HTTP request sent, awaiting response... 302 FOUND
Location: https://raw.githubusercontent.com/qiime2/distributions/dev/2024.10/amplicon/released/qiime2-amplicon-ubuntu-latest-conda.yml [following]
--2026-08-25 11:57:45--  https://raw.githubusercontent.com/qiime2/distributions/dev/2024.10/amplicon/released/qiime2-amplicon-ubuntu-latest-conda.yml
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 14947 (15K) [text/pl

In [ ]:
# QIIME 2 preflight: verify the installed command and DADA2 plugin before inputs are processed.

%%bash
set -euo pipefail
if [ "${BIOLOGY_ENABLED:-true}" = "false" ]; then
  echo "Biology disabled: skipping QIIME2 environment check."
  exit 0
fi

source /content/miniforge/etc/profile.d/conda.sh
conda env list | grep qiime2

conda activate qiime2-amplicon-2024.10

which qiime
qiime info
qiime dada2 denoise-paired --help | head -n 20


qiime2-amplicon-2024.10     /content/miniforge/envs/qiime2-amplicon-2024.10
/content/miniforge/envs/qiime2-amplicon-2024.10/bin/qiime
System versions
Python version: 3.10.14
QIIME 2 release: 2024.10
QIIME 2 version: 2024.10.1
q2cli version: 2024.10.1

Installed plugins
alignment: 2024.10.0
composition: 2024.10.0
cutadapt: 2024.10.0
dada2: 2024.10.0
deblur: 2024.10.0
demux: 2024.10.0
diversity: 2024.10.0
diversity-lib: 2024.10.0
emperor: 2024.10.0
feature-classifier: 2024.10.0
feature-table: 2024.10.0
fragment-insertion: 2024.10.0
longitudinal: 2024.10.0
metadata: 2024.10.0
phylogeny: 2024.10.0
quality-control: 2024.10.0
quality-filter: 2024.10.0
rescript: 2024.10.0
sample-classifier: 2024.10.0
stats: 0+unknown
taxa: 2024.10.0
types: 2024.10.0
vizard: 0.0.1.dev0
vsearch: 2024.10.0

Application config directory
/content/miniforge/envs/qiime2-amplicon-2024.10/var/q2cli

Config
Config Source: /content/miniforge/envs/qiime2-amplicon-2024.10/etc/qiime2_config.toml

Getting help
To get help w

In [ ]:
%%bash
set -euo pipefail
if [ "${BIOLOGY_ENABLED:-true}" = "false" ]; then
  echo "Biology disabled: skipping SRA-tools setup."
  exit 0
fi

source /content/miniforge/etc/profile.d/conda.sh

echo "Creating SRA tools environment if needed..."

if ! conda env list | grep -q "sra-tools-env"; then
  conda create -y \
    -n sra-tools-env \
    -c conda-forge \
    -c bioconda \
    sra-tools entrez-direct pigz pandas
fi

conda activate sra-tools-env

echo "Testing tools..."
which prefetch
which fasterq-dump
which esearch
which efetch

prefetch --version
fasterq-dump --version

echo "SRA tools environment is ready."


Creating SRA tools environment if needed...
Channels:
 - conda-forge
 - bioconda
Platform: linux-64
Solving environment: - done

## Package Plan ##

  environment location: /content/miniforge/envs/sra-tools-env

  added / updated specs:
    - entrez-direct
    - pandas
    - pigz
    - sra-tools


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    c-ares-1.34.8              |       hebe6cf0_2         223 KB  conda-forge
    curl-8.21.0                |       ha042cf0_5         190 KB  conda-forge
    entrez-direct-26.0         |       h1079eea_0        15.4 MB  bioconda
    ld_impl_linux-64-2.46.1    |default_hbd61a6d_102         728 KB  conda-forge
    libblas-3.11.0             |9_h4a7cf45_openblas          18 KB  conda-forge
    libcblas-3.11.0            |9_h0358290_openblas          18 KB  conda-forge
    libcurl-8.21.0             |       ha042cf0_5         473 KB  conda-forge
    l

WARNING conda.conda_pypi.main:notify_externally_managed_future(156): 
  Did you know? You can install many PyPI packages with conda
  using the conda-pypi beta. Get started:
    https://docs.conda.io/projects/conda/en/stable/new-features.html



In [ ]:
# The integration repository was cloned before loading config/preferences.yaml.

%%bash
set -euo pipefail
if [ "${BIOLOGY_ENABLED:-true}" = "false" ]; then
  echo "Biology disabled: skipping repository inspection."
  exit 0
fi

test -d "$REPO_DIR/.git"
cd "$REPO_DIR"
git rev-parse --short HEAD
ls -lh


total 48K
drwxr-xr-x 6 root root 4.0K Aug 25 12:08 AWS_runs_results
-rw-r--r-- 1 root root 1.1K Aug 25 12:08 colombian_runs_full.txt
-rw-r--r-- 1 root root  11K Aug 25 12:08 main.nf
drwxr-xr-x 2 root root 4.0K Aug 25 12:08 metadata_files
-rw-r--r-- 1 root root  16K Aug 25 12:08 metadata.tsv
-rw-r--r-- 1 root root 1020 Aug 25 12:08 nextflow.config
-rw-r--r-- 1 root root 1.3K Aug 25 12:08 README.md


In [ ]:
if not BIOLOGY_ENABLED:
    print("Biology disabled: using existing ML inputs; skipping metadata harmonization.")
else:
    ML_ROOT.mkdir(parents=True,exist_ok=True)
    INPUT_DIR=ML_ROOT/'input files'; INPUT_DIR.mkdir(parents=True,exist_ok=True)
    def find_drive_file(name):
        requested=Path(name)
        if requested.is_file(): return requested
        direct=DRIVE_ROOT/requested.name
        if direct.is_file(): return direct
        matches=sorted(p for p in DRIVE_ROOT.rglob(requested.name) if p.is_file())
        if not matches: raise FileNotFoundError(f'Missing {requested.name} under {DRIVE_ROOT}')
        print('Using:',matches[0]); return matches[0]
    shutil.copy2(find_drive_file(CONFIG['input']['phenotype_metadata']),PHENOTYPE_FILE)
    shutil.copy2(find_drive_file(CONFIG['input']['run_participant_map']),RUN_MAPPING_FILE)
    phen=pd.read_csv(PHENOTYPE_FILE,sep='\t',dtype=str).fillna('')
    def pid(x):
        x=str(x).strip()
        prefix=CONFIG['metadata']['participant_id_prefix']
        has_prefix=x.casefold().startswith(prefix.casefold())
        suffix=x[len(prefix):] if has_prefix else x
        matched_alias=False
        for alias in CONFIG['metadata']['participant_id_alias_prefixes']:
            if suffix.casefold().startswith(alias.casefold()):
                suffix=suffix[len(alias):]
                matched_alias=True
                break
        return prefix+suffix if has_prefix or matched_alias else x
    sample_id_column=CONFIG['metadata']['qiime_sample_id_column']; group_column=CONFIG['metadata']['group_column']
    run_id_column=CONFIG['input']['run_id_column']; run_participant_id_column=CONFIG['input']['run_participant_id_column']
    run_map=pd.read_excel(RUN_MAPPING_FILE,sheet_name=CONFIG['input']['run_participant_sheet'],dtype=str).fillna('')
    for frame, columns, label in [(run_map,{run_id_column,run_participant_id_column},'run-participant workbook'), (phen,{MODEL_ID_COLUMN,OUTCOME_COLUMN},'phenotype metadata')]:
        missing=columns-set(frame.columns)
        if missing: raise ValueError(f'{label} is missing required columns: {sorted(missing)}')
    run_map[sample_id_column]=run_map[run_id_column].astype(str).str.strip()
    run_map[MODEL_ID_COLUMN]=run_map[run_participant_id_column].map(pid)
    if (run_map[sample_id_column]=='').any() or (run_map[MODEL_ID_COLUMN]=='').any(): raise ValueError('The run-participant workbook contains blank run or participant identifiers')
    if run_map[sample_id_column].duplicated().any(): raise ValueError('The run-participant workbook contains duplicate run identifiers')
    phen[MODEL_ID_COLUMN]=phen[MODEL_ID_COLUMN].map(pid)
    if (phen[MODEL_ID_COLUMN]=='').any() or phen[MODEL_ID_COLUMN].duplicated().any(): raise ValueError('Phenotype metadata contains blank or duplicate participant identifiers')
    label_map={str(key).strip().casefold(): value for key,value in CONFIG['metadata']['label_map'].items()}
    phen[group_column]=phen[OUTCOME_COLUMN].astype(str).str.strip().str.casefold().map(label_map)
    unknown_outcomes=sorted(phen.loc[phen[group_column].isna(),OUTCOME_COLUMN].astype(str).unique())
    if unknown_outcomes: raise ValueError(f'Phenotype outcome values have no metadata.label_map entry: {unknown_outcomes[:10]}')
    missing_phenotype_ids=sorted(set(run_map[MODEL_ID_COLUMN])-set(phen[MODEL_ID_COLUMN]))
    if missing_phenotype_ids: raise ValueError(f'Run-participant workbook IDs missing from phenotype metadata ({len(missing_phenotype_ids)}): {missing_phenotype_ids[:10]}')
    source=CONFIG['input'].get('qiime_metadata_source')
    raw_path=None
    if source:
        configured_path=Path(source)
        if configured_path.is_file(): raw_path=configured_path
        elif (REPO_DIR/configured_path).is_file(): raw_path=REPO_DIR/configured_path
        else:
            try: raw_path=find_drive_file(configured_path.name)
            except FileNotFoundError: pass
    if raw_path:
        raw=pd.read_csv(raw_path,sep='\t',dtype=str).fillna('')
        if raw.empty: raise ValueError(f'QIIME metadata source is empty: {raw_path}')
        raw=raw.rename(columns={raw.columns[0]:sample_id_column})
        raw[sample_id_column]=raw[sample_id_column].astype(str).str.strip()
        if (raw[sample_id_column]=='').any() or raw[sample_id_column].duplicated().any(): raise ValueError('QIIME metadata source contains blank or duplicate sample IDs')
        raw=raw.drop(columns=[MODEL_ID_COLUMN,group_column,OUTCOME_COLUMN],errors='ignore')
        metadata_source=str(raw_path)
    else:
        raw=run_map[[sample_id_column]].copy()
        metadata_source='run-participant workbook fallback'
        if source: print(f'Configured QIIME metadata was not found ({raw_path}); using the workbook fallback.')
    merged=raw.merge(run_map[[sample_id_column,MODEL_ID_COLUMN]],on=sample_id_column,how='left',validate='one_to_one').merge(phen[[MODEL_ID_COLUMN,group_column,OUTCOME_COLUMN]],on=MODEL_ID_COLUMN,how='left',validate='many_to_one')
    missing_sample_ids=merged.loc[merged[MODEL_ID_COLUMN].isna(),sample_id_column].astype(str).tolist()
    if missing_sample_ids: raise ValueError(f'QIIME metadata sample IDs missing from the run-participant workbook ({len(missing_sample_ids)}): {missing_sample_ids[:10]}')
    if merged[group_column].isna().any(): raise RuntimeError('Group labels are unexpectedly missing after metadata validation')
    if merged[group_column].nunique()!=2: raise ValueError('Exactly two phenotype groups are required')
    QIIME_METADATA=REPO_DIR/'metadata.tsv'; merged.to_csv(QIIME_METADATA,sep='\t',index=False)
    merged[sample_id_column].to_csv(REPO_DIR/'colombian_runs_full.txt',index=False,header=False)
    print('Unified metadata:',QIIME_METADATA, f'({metadata_source})'); print(merged[group_column].value_counts())


Unified metadata: /content/project/CSthesis/metadata.tsv
group
healthy       73
depressive    15
Name: count, dtype: int64


In [ ]:
# Download the full dataset locally into /content

%%bash
set -euo pipefail
if [ "${BIOLOGY_ENABLED:-true}" = "false" ]; then
  echo "Biology disabled: skipping FASTQ download."
  exit 0
fi

# Restore and validate DADA2 before deciding whether FASTQs are needed.
RESTORE_DIR=/content/restore_from_drive
RESTORE_FLAG="${RESTORE_DIR}/restored.flag"
rm -f "$RESTORE_FLAG"
for archive in "${ARCHIVE_DIR}/${RUN_NAME}_${BRANCH_KEY}_results.tar.gz" "${ARCHIVE_DIR}/${RUN_NAME}_asv_results.tar.gz" "${ARCHIVE_DIR}/${RUN_NAME}_dada2_results.tar.gz"; do
  [ -f "$archive" ] || continue
  rm -rf "$RESTORE_DIR"
  mkdir -p "$RESTORE_DIR"
  if ! python - "$archive" "$RESTORE_DIR" <<'PY'
from pathlib import Path
import sys, tarfile
archive, destination = map(Path, sys.argv[1:])
root = destination.resolve()
with tarfile.open(archive, 'r:gz') as tar:
    members = tar.getmembers()
    for member in members:
        target = (destination / member.name).resolve()
        if not target.is_relative_to(root) or member.issym() or member.islnk():
            raise RuntimeError(f'Unsafe archive member: {member.name}')
    tar.extractall(destination, members=members)
PY
  then
    echo "Could not safely extract $archive; trying the next candidate."
    continue
  fi
  saved="${RESTORE_DIR}/results/colombian_full"
  valid=true
  for relative_path in dada2/table.qza dada2/rep-seqs.qza dada2/denoising-stats.qza; do
    artifact="${saved}/${relative_path}"
    if [ ! -f "$artifact" ] || ! "$QIIME_BIN" tools peek "$artifact"; then
      echo "Invalid or missing restored DADA2 artifact: $artifact"
      valid=false
      break
    fi
  done
  if [ "$valid" = true ]; then
    printf '%s\n' "$archive" > "$RESTORE_FLAG"
    echo "Validated DADA2 archive before FASTQ download: $archive"
    break
  fi
done

if [ -f "$RESTORE_FLAG" ]; then
  echo "Valid DADA2 archive found: skipping FASTQ download."
  exit 0
fi

source /content/miniforge/etc/profile.d/conda.sh
conda activate sra-tools-env

cd /content/project

mkdir -p "$READS_DIR"
mkdir -p /content/project/sra_cache
mkdir -p /content/project/tmp/fasterq

while read acc; do
  echo "========================================"
  echo "Processing $acc"
  echo "========================================"

  if [ -f "$READS_DIR/${acc}_1.fastq.gz" ] && [ -f "$READS_DIR/${acc}_2.fastq.gz" ]; then
    echo "$acc already exists, skipping."
    continue
  fi

  prefetch "$acc" \
    --max-size 100G \
    -O /content/project/sra_cache

  sra_file=$(find /content/project/sra_cache -name "${acc}.sra" | head -n 1)

  if [ -z "$sra_file" ]; then
    echo "Could not find SRA file for $acc"
    exit 1
  fi

  fasterq-dump "$sra_file" \
    --split-files \
    --threads 2 \
    --temp /content/project/tmp/fasterq \
    -O "$READS_DIR"

  pigz -p 2 "$READS_DIR/${acc}_1.fastq"
  pigz -p 2 "$READS_DIR/${acc}_2.fastq"

  rm -rf /content/project/tmp/fasterq/*

done < "$REPO_DIR/colombian_runs_full.txt"

echo
echo "Download complete."
echo "Forward files:"
find "$READS_DIR" -type f -name "*_1.fastq.gz" | wc -l
echo "Reverse files:"
find "$READS_DIR" -type f -name "*_2.fastq.gz" | wc -l
du -sh "$READS_DIR"


Processing SRR25474187
2026-08-25T12:20:15 prefetch.3.4.1: 1) Resolving 'SRR25474187'...
2026-08-25T12:20:16 prefetch.3.4.1: Current preference is set to retrieve SRA Normalized Format files with full base quality scores
2026-08-25T12:20:16 prefetch.3.4.1: 1) Downloading 'SRR25474187'...
2026-08-25T12:20:16 prefetch.3.4.1:  SRA Normalized Format file is being retrieved
2026-08-25T12:20:16 prefetch.3.4.1:  Downloading via HTTPS...
2026-08-25T12:20:17 prefetch.3.4.1:  HTTPS download succeed
2026-08-25T12:20:17 prefetch.3.4.1:  'SRR25474187' is valid: 33510453 bytes were streamed from 33497467
2026-08-25T12:20:17 prefetch.3.4.1: 1) 'SRR25474187' was downloaded successfully
2026-08-25T12:20:17 prefetch.3.4.1: 1) Resolving 'SRR25474187's dependencies...
2026-08-25T12:20:17 prefetch.3.4.1: 'SRR25474187' has 0 unresolved dependencies
Processing SRR25474186
2026-08-25T12:20:21 prefetch.3.4.1: 1) Resolving 'SRR25474186'...
2026-08-25T12:20:21 prefetch.3.4.1: Current preference is set to retriev

spots read      : 101,618
reads read      : 203,236
reads written   : 203,236
spots read      : 58,361
reads read      : 116,722
reads written   : 116,722
spots read      : 83,167
reads read      : 166,334
reads written   : 166,334
spots read      : 82,988
reads read      : 165,976
reads written   : 165,976
spots read      : 84,529
reads read      : 169,058
reads written   : 169,058
spots read      : 73,245
reads read      : 146,490
reads written   : 146,490
spots read      : 83,438
reads read      : 166,876
reads written   : 166,876
spots read      : 92,971
reads read      : 185,942
reads written   : 185,942
spots read      : 100,996
reads read      : 201,992
reads written   : 201,992
spots read      : 93,149
reads read      : 186,298
reads written   : 186,298
spots read      : 92,547
reads read      : 185,094
reads written   : 185,094
spots read      : 98,927
reads read      : 197,854
reads written   : 197,854
spots read      : 82,477
reads read      : 164,954
reads written   : 164,9

In [ ]:
# Validate that every data has both FASTQs

%%bash
set -euo pipefail
if [ "${BIOLOGY_ENABLED:-true}" = "false" ]; then
  echo "Biology disabled: skipping FASTQ validation."
  exit 0
fi

if [ -f /content/restore_from_drive/restored.flag ]; then
  echo "Valid DADA2 archive restored: skipping FASTQ validation."
  exit 0
fi

python - <<'PY'
from pathlib import Path
import os
import pandas as pd

meta = pd.read_csv(Path(os.environ['REPO_DIR']) / 'metadata.tsv', sep="\t")
samples = meta["sample-id"].astype(str).tolist()

fastq_dir = Path(os.environ['READS_DIR'])

missing = []

for sample in samples:
    r1 = fastq_dir / f"{sample}_1.fastq.gz"
    r2 = fastq_dir / f"{sample}_2.fastq.gz"

    if not r1.exists() or not r2.exists():
        missing.append(sample)

print("Metadata samples:", len(samples))
print("Missing pairs:", len(missing))

if missing:
    print("Missing:")
    print("\n".join(missing))
    raise SystemExit(1)

print("All metadata samples have R1/R2 FASTQs.")
PY


Metadata samples: 88
Missing pairs: 0
All metadata samples have R1/R2 FASTQs.


In [ ]:
# prepare the SILVA classifier for the Nextflow run

%%bash
set -euo pipefail
if [ "${BIOLOGY_ENABLED:-true}" = "false" ]; then
  echo "Biology disabled: skipping SILVA download/check."
  exit 0
fi

source /content/miniforge/etc/profile.d/conda.sh
conda activate qiime2-amplicon-2024.10

if [ ! -f /content/project/refs/silva-138-99-nb-classifier.qza ]; then
  mkdir -p /content/project/refs
  wget -O /content/project/refs/silva-138-99-nb-classifier.qza \
    https://data.qiime2.org/classifiers/sklearn-1.4.2/silva/silva-138-99-nb-classifier.qza
fi

ls -lh /content/project/refs/silva-138-99-nb-classifier.qza
qiime tools peek /content/project/refs/silva-138-99-nb-classifier.qza


-rw-r--r-- 1 root root 209M Jun 24  2024 /content/project/refs/silva-138-99-nb-classifier.qza
UUID:        70b4b5f4-8fce-40bd-b508-afacbc12a5ed
Type:        TaxonomicClassifier
Data format: TaxonomicClassiferTemporaryPickleDirFmt


--2026-08-25 12:28:05--  https://data.qiime2.org/classifiers/sklearn-1.4.2/silva/silva-138-99-nb-classifier.qza
Resolving data.qiime2.org (data.qiime2.org)... 54.200.1.12
Connecting to data.qiime2.org (data.qiime2.org)|54.200.1.12|:443... connected.
HTTP request sent, awaiting response... 302 FOUND
Location: https://s3-us-west-2.amazonaws.com/qiime2-data/classifiers/sklearn-1.4.2/silva/silva-138-99-nb-classifier.qza [following]
--2026-08-25 12:28:06--  https://s3-us-west-2.amazonaws.com/qiime2-data/classifiers/sklearn-1.4.2/silva/silva-138-99-nb-classifier.qza
Resolving s3-us-west-2.amazonaws.com (s3-us-west-2.amazonaws.com)... 52.92.227.128, 52.92.179.80, 52.92.248.24, ...
Connecting to s3-us-west-2.amazonaws.com (s3-us-west-2.amazonaws.com)|52.92.227.128|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 218245868 (208M) [binary/octet-stream]
Saving to: ‘/content/project/refs/silva-138-99-nb-classifier.qza’

     0K .......... .......... .......... .......... .

## 2. Parameterized Nextflow run

DADA2 is run only when no validated archive is available; the selected ASV or OTU branch always continues through Nextflow.


In [ ]:
if not BIOLOGY_ENABLED:
    print("Biology disabled: skipping SILVA validation and Nextflow configuration.")
else:
    # silva classifier check
    from pathlib import Path

    CLASSIFIER = Path(CONFIG["taxonomy"]["classifier"])
    if not CLASSIFIER.is_absolute():
        CLASSIFIER = (PROJECT_ROOT / CLASSIFIER).resolve()
    if not CLASSIFIER.is_file():
        raise FileNotFoundError(f"SILVA classifier not found: {CLASSIFIER}")
    print("Using classifier:", CLASSIFIER)

    # nextflow run config for google colab
    q=CONFIG['quality']; p=CONFIG['primers']
    NEXTFLOW_CONFIG=WORKFLOW_DIR/'nextflow.colab.config'
    config_lines=[
    "workDir = '/content/nxf_work'",
    "process {", "    executor = 'local'", "    cpus = 8", "    memory = '36 GB'", "    maxForks = 1", "    stageInMode = 'copy'", "    beforeScript = '''", "    source /content/miniforge/etc/profile.d/conda.sh", "    conda activate qiime2-amplicon-2024.10", "    mkdir -p .numba_cache .mpl_cache .qiime2_cache", "    export NUMBA_CACHE_DIR=\"$PWD/.numba_cache\"", "    export MPLCONFIGDIR=\"$PWD/.mpl_cache\"", "    export QIIME2_CACHE_DIR=\"$PWD/.qiime2_cache\"", "    '''", "}",
    "timeline { enabled = true; file = '/content/project/results/pipeline_info/timeline.html' }", "report { enabled = true; file = '/content/project/results/pipeline_info/report.html' }", "trace { enabled = true; file = '/content/project/results/pipeline_info/trace.txt'; fields = 'task_id,name,status,exit,realtime,duration,%cpu,%mem,rss' }",
    "params {", f"    metadata_url = '{QIIME_METADATA}'", f"    classifier_url = '{CLASSIFIER}'", f"    outdir = '{RESULTS_ROOT}'", f"    branch = '{NEXTFLOW_BRANCH}'", f"    primer_f = '{p['forward']}'", f"    primer_r = '{p['reverse']}'", f"    trunc_len_f = {q['trunc_len_f']}", f"    trunc_len_r = {q['trunc_len_r']}", f"    trim_left_f = {q['trim_left_f']}", f"    trim_left_r = {q['trim_left_r']}", f"    max_ee = {q['max_expected_errors']}", f"    min_quality = {q['min_quality']}", f"    error_rate = {q['error_rate']}", f"    sampling_depth = {q['sampling_depth']}", "}"]
    NEXTFLOW_CONFIG.write_text('\n'.join(config_lines)+'\n')
    print(NEXTFLOW_CONFIG.read_text())


Using classifier: /content/project/refs/silva-138-99-nb-classifier.qza
workDir = '/content/nxf_work'
process {
    executor = 'local'
    cpus = 8
    memory = '36 GB'
    maxForks = 1
    stageInMode = 'copy'
    beforeScript = '''
    source /content/miniforge/etc/profile.d/conda.sh
    conda activate qiime2-amplicon-2024.10
    mkdir -p .numba_cache .mpl_cache .qiime2_cache
    export NUMBA_CACHE_DIR="$PWD/.numba_cache"
    export MPLCONFIGDIR="$PWD/.mpl_cache"
    export QIIME2_CACHE_DIR="$PWD/.qiime2_cache"
    '''
}
timeline { enabled = true; file = '/content/project/results/pipeline_info/timeline.html' }
report { enabled = true; file = '/content/project/results/pipeline_info/report.html' }
trace { enabled = true; file = '/content/project/results/pipeline_info/trace.txt'; fields = 'task_id,name,status,exit,realtime,duration,%cpu,%mem,rss' }
params {
    metadata_url = '/content/project/CSthesis/metadata.tsv'
    classifier_url = '/content/project/refs/silva-138-99-nb-classifier.q

In [ ]:
# Adopt the DADA2 archive that was validated before FASTQ download.
# Nextflow will still run the selected ASV or OTU branch from these artifacts.

RESTORED_FROM_ARCHIVE = False
RESTORE_DIR = Path('/content/restore_from_drive')
RESTORE_FLAG = RESTORE_DIR / 'restored.flag'
SAVED_RESULTS = RESTORE_DIR / 'results/colombian_full'
DADA2_OUTPUTS = (
    'dada2/table.qza',
    'dada2/rep-seqs.qza',
    'dada2/denoising-stats.qza',
)

if not BIOLOGY_ENABLED:
    print('Biology disabled: skipping DADA2 restoration.')
elif not RESTORE_FLAG.is_file():
    print('No validated DADA2 archive found; Nextflow will denoise the reads.')
else:
    restored_outputs = [SAVED_RESULTS / relative_path for relative_path in DADA2_OUTPUTS]
    try:
        if not all(path.is_file() for path in restored_outputs):
            raise FileNotFoundError('The validated archive no longer has all DADA2 outputs.')
        for artifact in restored_outputs:
            subprocess.run([str(QIIME_BIN), 'tools', 'peek', str(artifact)], check=True)
        if RESULTS_ROOT.exists():
            shutil.rmtree(RESULTS_ROOT)
        shutil.copytree(SAVED_RESULTS, RESULTS_ROOT)
        RESTORED_FROM_ARCHIVE = True
        print('Validated DADA2 results restored from Drive; Nextflow will run the selected branch.')
        print('Archive:', RESTORE_FLAG.read_text().strip())
    except (FileNotFoundError, OSError, subprocess.CalledProcessError) as error:
        print(f'Could not use the validated DADA2 archive ({error}); Nextflow will denoise the reads.')
os.environ['RESTORED_FROM_ARCHIVE'] = str(RESTORED_FROM_ARCHIVE).lower()


DADA2 results restored from Drive; Nextflow will be skipped.
/content/project/results/colombian_full/dada2/table.qza
/content/project/results/colombian_full/dada2/rep-seqs.qza
/content/project/results/colombian_full/dada2/denoising-stats.qza


In [ ]:
if not BIOLOGY_ENABLED:
    print("Biology disabled: skipping the DADA2 run.")
else:
    RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
    Path('/content/nxf_work').mkdir(parents=True, exist_ok=True)
    command = [
        'nextflow', '-C', str(NEXTFLOW_CONFIG), 'run', 'main.nf',
        '--metadata_url', str(QIIME_METADATA),
        '--outdir', str(RESULTS_ROOT),
        '--classifier_url', str(CLASSIFIER),
        '--branch', BRANCH,
        '--otu_similarity', str(CONFIG['otu']['similarity']),
        '-work-dir', '/content/nxf_work', '-resume',
    ]
    if RESTORED_FROM_ARCHIVE:
        command.extend(['--dada2_dir', str(RESULTS_ROOT / 'dada2')])
    else:
        command.extend(['--reads_dir', CONFIG['reads']['directory']])
    subprocess.run(command, cwd=WORKFLOW_DIR, check=True)
    for relative_path in DADA2_OUTPUTS:
        path = RESULTS_ROOT / relative_path
        if not path.is_file():
            raise FileNotFoundError(path)
    print('Selected branch completed through Nextflow; DADA2 source:', 'Drive archive' if RESTORED_FROM_ARCHIVE else 'new denoising')


Skipping Nextflow run because valid DADA2 outputs were restored from Drive.


## 3. Verify selected Nextflow branch and prepare downstream inputs


In [ ]:
# Nextflow has already completed the selected ASV or OTU branch.
# This cell verifies its published artifacts and defines paths for later exports.
if not BIOLOGY_ENABLED:
    print("Biology disabled: skipping ASV/OTU output verification.")
else:
    def qrun(*args):
        cmd=[str(QIIME_BIN),*map(str,args)]; print(' '.join(cmd)); subprocess.run(cmd,check=True)
    if BRANCH_KEY=='asv':
        BRANCH_DIR=RESULTS_ROOT; TABLE=RESULTS_ROOT/'dada2/table.qza'; REP_SEQS=RESULTS_ROOT/'dada2/rep-seqs.qza'
    else:
        BRANCH_DIR=RESULTS_ROOT/'otu'; TABLE=BRANCH_DIR/'table.qza'; REP_SEQS=BRANCH_DIR/'rep-seqs.qza'
    TAXONOMY=RESULTS_ROOT/'taxonomy/taxonomy.qza'; TREE=RESULTS_ROOT/'phylogeny/rooted-tree.qza'
    DIVERSITY_DIR=RESULTS_ROOT/'diversity/core-metrics-results'
    CHAO1=RESULTS_ROOT/'alpha_diversity/chao1-vector.qza'
    required_outputs=[TABLE, REP_SEQS, TAXONOMY, TREE, DIVERSITY_DIR/'shannon_vector.qza', CHAO1]
    missing=[str(path) for path in required_outputs if not path.is_file()]
    if missing:
        raise FileNotFoundError('Nextflow did not produce selected-branch outputs:\n' + '\n'.join(missing))
    print('Selected branch completed:',BRANCH)


Selected branch completed: asv


In [ ]:
# Execute the reusable diversity-plot helper after selected-branch verification.

if BIOLOGY_ENABLED:
    plot_diversity_outputs()
else:
    print("Biology disabled: skipping ASV/OTU diversity plots.")


/content/miniforge/envs/qiime2-amplicon-2024.10/bin/qiime tools export --input-path /content/project/results/colombian_full/diversity/core-metrics-results/shannon_vector.qza --output-path /content/project/results/colombian_full/diversity/exports/alpha_shannon
/content/miniforge/envs/qiime2-amplicon-2024.10/bin/qiime tools export --input-path /content/project/results/colombian_full/diversity/core-metrics-results/observed_features_vector.qza --output-path /content/project/results/colombian_full/diversity/exports/alpha_observed_features
/content/miniforge/envs/qiime2-amplicon-2024.10/bin/qiime tools export --input-path /content/project/results/colombian_full/diversity/core-metrics-results/evenness_vector.qza --output-path /content/project/results/colombian_full/diversity/exports/alpha_pielou_evenness
/content/miniforge/envs/qiime2-amplicon-2024.10/bin/qiime tools export --input-path /content/project/results/colombian_full/diversity/core-metrics-results/faith_pd_vector.qza --output-path /c

In [ ]:
# Export the verified selected-branch feature table and taxonomy for ML preparation.

if not BIOLOGY_ENABLED:
    print("Biology disabled: skipping QIIME export preparation.")
else:
    EXPORT_ROOT=ML_ROOT/'qiime_exports'
    if EXPORT_ROOT.exists(): shutil.rmtree(EXPORT_ROOT)
    EXPORT_ROOT.mkdir(parents=True)
    label='asv' if BRANCH_KEY=='asv' else 'otu'
    branch_table_dir=EXPORT_ROOT/f'{label}_table'; branch_tax_dir=EXPORT_ROOT/f'{label}_taxonomy'
    qrun('tools','export','--input-path',TABLE,'--output-path',branch_table_dir)
    qrun('tools','export','--input-path',TAXONOMY,'--output-path',branch_tax_dir)
    subprocess.run([str(BIOM_BIN),'convert','-i',str(branch_table_dir/'feature-table.biom'),'-o',str(branch_table_dir/'feature-table.tsv'),'--to-tsv'],check=True)


/content/miniforge/envs/qiime2-amplicon-2024.10/bin/qiime tools export --input-path /content/project/results/colombian_full/dada2/table.qza --output-path /content/week4_dataset1/qiime_exports/asv_table
/content/miniforge/envs/qiime2-amplicon-2024.10/bin/qiime tools export --input-path /content/project/results/colombian_full/taxonomy/taxonomy.qza --output-path /content/week4_dataset1/qiime_exports/asv_taxonomy


## 4. Selected-branch abundance matrix and quality checks


In [ ]:
if not BIOLOGY_ENABLED:
    print("Biology disabled: skipping selected-branch matrix construction.")
else:
    BRANCH_KEY='otu' if CONFIG['run']['branch']=='otu' else 'asv'
    # Build the selected ASV or OTU matrix and map run IDs to participant IDs.

    from pathlib import Path
    import math

    import numpy as np
    import pandas as pd


    WEEK4_ROOT=Path(CONFIG['run']['ml_dir'])
    EXPORT_ROOT = WEEK4_ROOT / "qiime_exports"
    METADATA_FILE = PHENOTYPE_FILE
    MATRIX_ROOT = WEEK4_ROOT / "matrices"
    RUN_METADATA_FILE = RUN_MAPPING_FILE
    if not RUN_METADATA_FILE.is_file():
        raise FileNotFoundError(f'Run-to-participant workbook not found: {RUN_METADATA_FILE}')

    PSEUDOCOUNT=float(CONFIG['ml'].get('pseudocount',0.5))
    MIN_PREVALENCE_FRACTION=float(CONFIG['filtering'].get('min_prevalence',0.10))
    MIN_TOTAL_COUNT=int(CONFIG['filtering'].get('min_total_abundance',10))

    MATRIX_ROOT.mkdir(
        parents=True,
        exist_ok=True,
    )


    def participant_id_from_value(value: object) -> str:
        """
        Normalize participant IDs according to config/preferences.yaml.
        """
        value = str(value).strip()

        prefix = CONFIG['metadata']['participant_id_prefix']
        if value.startswith(prefix):
            return value

        for alias in CONFIG['metadata']['participant_id_alias_prefixes']:
            if value.startswith(alias):
                return prefix + value[len(alias):]

        return value


    # ------------------------------------------------------------
    # Build the SRR-to-participant mapping
    # ------------------------------------------------------------

    run_metadata = pd.read_excel(
        RUN_METADATA_FILE,
        sheet_name=CONFIG['input']['run_participant_sheet'],
        dtype=str,
    )

    run_id_column = CONFIG['input']['run_id_column']
    run_participant_id_column = CONFIG['input']['run_participant_id_column']
    required_run_columns = {run_id_column, run_participant_id_column}

    missing_run_columns = (
        required_run_columns - set(run_metadata.columns)
    )

    if missing_run_columns:
        raise ValueError(
            f"Run metadata is missing columns: "
            f"{sorted(missing_run_columns)}"
        )

    run_metadata[run_id_column] = (
        run_metadata[run_id_column]
        .astype(str)
        .str.strip()
    )

    run_metadata[MODEL_ID_COLUMN] = (
        run_metadata[run_participant_id_column]
        .map(participant_id_from_value)
    )

    if run_metadata[run_id_column].duplicated().any():
        raise ValueError(
            "Duplicate sequencing Run IDs found."
        )

    if run_metadata[MODEL_ID_COLUMN].duplicated().any():
        raise ValueError(
            "Duplicate participant IDs found in run metadata."
        )

    run_to_participant = dict(
        zip(
            run_metadata[run_id_column],
            run_metadata[MODEL_ID_COLUMN],
        )
    )

    run_metadata.loc[
        :,
        [run_id_column, run_participant_id_column, MODEL_ID_COLUMN],
    ].to_csv(
        MATRIX_ROOT / "run_to_participant_mapping.tsv",
        sep="\t",
        index=False,
    )

    print("Run-to-participant mapping loaded:")
    print("Mapping rows:", len(run_to_participant))
    print(
        run_metadata[
            [run_id_column, run_participant_id_column, MODEL_ID_COLUMN]
        ]
        .head()
        .to_string(index=False)
    )


    def read_biom_tsv(path: Path) -> pd.DataFrame:
        """
        Read a TSV exported from BIOM.

        The header normally begins with '#OTU ID'.
        """
        lines = path.read_text().splitlines()

        header_index = next(
            (
                index
                for index, line in enumerate(lines)
                if line.startswith("#OTU ID")
            ),
            None,
        )

        if header_index is None:
            raise ValueError(
                f"Could not find '#OTU ID' header in {path}"
            )

        return pd.read_csv(
            path,
            sep="\t",
            skiprows=header_index,
        )


    def resolve_sample_id(
        raw_sample_id: object,
        run_mapping: dict[str, str],
    ) -> str:
        """
        Resolve a QIIME sample ID to the participant ID used
        by the ML metadata.
        """
        raw_sample_id = str(raw_sample_id).strip()

        # Main case for this dataset: SRR run ID.
        if raw_sample_id in run_mapping:
            return run_mapping[raw_sample_id]

        # Fallbacks for legacy sample-ID formats.
        return participant_id_from_value(raw_sample_id)


    def load_branch_inputs(
        table_path: Path,
        taxonomy_path: Path,
        metadata: pd.DataFrame,
        run_mapping: dict[str, str],
    ):
        table = read_biom_tsv(table_path)

        taxonomy = pd.read_csv(
            taxonomy_path,
            sep="\t",
        )

        feature_id_column = table.columns[0]
        sample_columns = list(table.columns[1:])

        table[feature_id_column] = (
            table[feature_id_column]
            .astype(str)
            .str.strip()
        )

        if table[feature_id_column].duplicated().any():
            raise ValueError(
                f"Duplicate feature IDs found in {table_path}"
            )

        counts = table.set_index(feature_id_column)[sample_columns]
        counts = counts.apply(pd.to_numeric, errors="raise")
        counts = counts.T

        original_sample_ids = (
            counts.index.astype(str).tolist()
        )

        normalized_sample_ids = [
            resolve_sample_id(
                sample_id,
                run_mapping,
            )
            for sample_id in original_sample_ids
        ]

        unresolved_run_ids = [
            sample_id
            for sample_id in original_sample_ids
            if (
                sample_id not in run_mapping
                and sample_id.startswith("SRR")
            )
        ]

        if unresolved_run_ids:
            raise ValueError(
                "Some SRR sample IDs could not be mapped to participants:\n"
                f"{unresolved_run_ids}"
            )

        if len(set(normalized_sample_ids)) != len(
            normalized_sample_ids
        ):
            raise ValueError(
                f"Sample IDs became duplicated after mapping "
                f"for {table_path}"
            )

        counts.index = normalized_sample_ids

        metadata_ids = (
            metadata[MODEL_ID_COLUMN]
            .astype(str)
            .str.strip()
            .tolist()
        )

        missing_from_metadata = (
            set(counts.index) - set(metadata_ids)
        )

        missing_from_table = (
            set(metadata_ids) - set(counts.index)
        )

        if missing_from_metadata or missing_from_table:
            raise ValueError(
                f"Sample mismatch for {table_path}\n"
                f"Missing from metadata: "
                f"{sorted(missing_from_metadata)}\n"
                f"Missing from table: "
                f"{sorted(missing_from_table)}"
            )

        # Reorder the count table to exactly match metadata order.
        counts = counts.loc[metadata_ids]

        taxonomy.columns = (
            taxonomy.columns.astype(str).str.strip()
        )

        required_taxonomy_columns = {
            "Feature ID",
            "Taxon",
        }

        missing_taxonomy_columns = (
            required_taxonomy_columns
            - set(taxonomy.columns)
        )

        if missing_taxonomy_columns:
            raise ValueError(
                f"Taxonomy file is missing columns: "
                f"{sorted(missing_taxonomy_columns)}"
            )

        taxonomy["Feature ID"] = (
            taxonomy["Feature ID"]
            .astype(str)
            .str.strip()
        )

        taxonomy["Taxon"] = (
            taxonomy["Taxon"]
            .fillna("Unassigned")
            .astype(str)
            .str.strip()
        )

        taxonomy_lookup = (
            taxonomy
            .drop_duplicates("Feature ID")
            .set_index("Feature ID")["Taxon"]
            .to_dict()
        )

        feature_metadata = pd.DataFrame(
            {
                "feature_id": counts.columns.astype(str),
                "taxon": [
                    taxonomy_lookup.get(
                        feature_id,
                        "Unassigned",
                    )
                    for feature_id in counts.columns
                ],
            }
        )

        return counts, feature_metadata


    def build_branch(
        branch_name: str,
        table_path: Path,
        taxonomy_path: Path,
        metadata: pd.DataFrame,
        run_mapping: dict[str, str],
    ):
        counts, feature_metadata = load_branch_inputs(
            table_path=table_path,
            taxonomy_path=taxonomy_path,
            metadata=metadata,
            run_mapping=run_mapping,
        )

        if (counts < 0).any().any():
            raise ValueError(
                f"Negative counts found in {branch_name}"
            )

        sample_depths = counts.sum(axis=1)

        if (sample_depths <= 0).any():
            raise ValueError(
                f"Zero-read samples found in {branch_name}"
            )

        min_prevalence_samples = max(
            1,
            math.ceil(
                MIN_PREVALENCE_FRACTION * len(counts)
            ),
        )

        prevalence = counts.gt(0).sum(axis=0)
        total_counts = counts.sum(axis=0)

        keep_features = (
            (prevalence >= min_prevalence_samples)
            & (total_counts >= MIN_TOTAL_COUNT)
        )

        filtered_counts = counts.loc[:, keep_features].copy()

        if filtered_counts.shape[1] == 0:
            raise ValueError(
                f"No features survived filtering for {branch_name}"
            )

        # Relative abundance.
        relative_abundance = (
            filtered_counts
            .div(filtered_counts.sum(axis=1), axis=0)
        )

        # CLR transformation.
        log_counts = np.log(
            filtered_counts.to_numpy(dtype=float)
            + PSEUDOCOUNT
        )

        clr_values = (
            log_counts
            - log_counts.mean(
                axis=1,
                keepdims=True,
            )
        )

        clr = pd.DataFrame(
            clr_values,
            index=filtered_counts.index,
            columns=filtered_counts.columns,
        )

        if not np.allclose(
            clr.mean(axis=1).to_numpy(),
            0.0,
            atol=1e-10,
        ):
            raise RuntimeError(
                f"CLR validation failed for {branch_name}"
            )

        feature_metadata = (
            feature_metadata
            .set_index("feature_id")
            .loc[filtered_counts.columns]
            .reset_index()
        )

        feature_metadata["prevalence_samples"] = (
            prevalence
            .loc[filtered_counts.columns]
            .to_numpy()
        )

        feature_metadata["total_count"] = (
            total_counts
            .loc[filtered_counts.columns]
            .to_numpy()
        )

        feature_metadata["branch"] = branch_name

        branch_dir = MATRIX_ROOT / branch_name
        branch_dir.mkdir(
            parents=True,
            exist_ok=True,
        )

        def save_matrix(
            matrix: pd.DataFrame,
            filename: str,
        ):
            output = matrix.copy()

            output.insert(
                0,
                MODEL_ID_COLUMN,
                output.index,
            )

            output.to_csv(
                branch_dir / filename,
                sep="\t",
                index=False,
            )

        save_matrix(
            filtered_counts,
            "counts.tsv",
        )

        save_matrix(
            relative_abundance,
            "relative_abundance.tsv",
        )

        save_matrix(
            clr,
            "clr.tsv",
        )

        feature_metadata.to_csv(
            branch_dir / "feature_metadata.tsv",
            sep="\t",
            index=False,
        )

        return {
            "branch": branch_name,
            "input_features": counts.shape[1],
            "retained_features": filtered_counts.shape[1],
            "sample_count": counts.shape[0],
            "minimum_prevalence_samples": (
                min_prevalence_samples
            ),
            "minimum_sample_depth": int(
                sample_depths.min()
            ),
            "median_sample_depth": float(
                sample_depths.median()
            ),
            "maximum_sample_depth": int(
                sample_depths.max()
            ),
        }


    metadata = pd.read_csv(
        METADATA_FILE,
        sep="\t",
    )

    required_metadata_columns = {MODEL_ID_COLUMN, OUTCOME_COLUMN, *DIET_COLUMNS}

    missing_metadata_columns = (
        required_metadata_columns
        - set(metadata.columns)
    )

    if missing_metadata_columns:
        raise ValueError(
            f"Metadata is missing columns: "
            f"{sorted(missing_metadata_columns)}"
        )

    if metadata[MODEL_ID_COLUMN].duplicated().any():
        raise ValueError(
            "Duplicate participant IDs found in metadata."
        )

    metadata.to_csv(
        MATRIX_ROOT / "model_metadata.tsv",
        sep="\t",
        index=False,
    )

    summary_rows=[build_branch(branch_name=BRANCH_KEY,table_path=EXPORT_ROOT/("asv_table" if BRANCH_KEY=="asv" else "otu_table")/"feature-table.tsv",taxonomy_path=EXPORT_ROOT/("asv_taxonomy" if BRANCH_KEY=="asv" else "otu_taxonomy")/"taxonomy.tsv",metadata=metadata,run_mapping=run_to_participant)]
    summary=pd.DataFrame(summary_rows)
    summary.to_csv(MATRIX_ROOT/"matrix_build_summary.tsv",sep="\t",index=False)
    print(summary.to_string(index=False))


Run-to-participant mapping loaded:
Mapping rows: 88
        Run Participant_id participant_id
SRR25474187         MIC002     sub-MIC002
SRR25474186         MIC003     sub-MIC003
SRR25474211         MIC005     sub-MIC005
SRR25474200         MIC008     sub-MIC008
SRR25474189         MIC009     sub-MIC009
branch  input_features  retained_features  sample_count  minimum_prevalence_samples  minimum_sample_depth  median_sample_depth  maximum_sample_depth
   asv            6144                511            88                           9                 17900              40181.0                 84036


In [ ]:
if not BIOLOGY_ENABLED:
    print("Biology disabled: using existing ML matrices; skipping matrix construction.")
else:
    BRANCH_KEY='otu' if CONFIG['run']['branch']=='otu' else 'asv'
    from pathlib import Path

    import numpy as np
    import pandas as pd


    WEEK4_ROOT=Path(CONFIG['run']['ml_dir'])
    EXPORT_ROOT = WEEK4_ROOT / "qiime_exports"
    MATRIX_ROOT = WEEK4_ROOT / "matrices"
    METADATA_FILE = PHENOTYPE_FILE


    def read_biom_tsv(path: Path) -> pd.DataFrame:
        """
        Read a TSV exported from BIOM.
        """
        lines = path.read_text().splitlines()

        header_index = next(
            (
                index
                for index, line in enumerate(lines)
                if line.startswith("#OTU ID")
            ),
            None,
        )

        if header_index is None:
            raise ValueError(
                f"Could not find '#OTU ID' header in {path}"
            )

        return pd.read_csv(
            path,
            sep="\t",
            skiprows=header_index,
        )


    def load_matrix(path: Path) -> pd.DataFrame:
        """
        Load one generated modeling matrix.
        """
        matrix = pd.read_csv(
            path,
            sep="\t",
        )

        if MODEL_ID_COLUMN not in matrix.columns:
            raise ValueError(
                f"Missing {MODEL_ID_COLUMN} column in {path}"
            )

        return matrix.set_index(MODEL_ID_COLUMN)


    def find_feature_id_column(
        dataframe: pd.DataFrame,
    ) -> str:
        """
        Detect the feature-ID column despite small naming differences.
        """
        aliases = {
            "feature_id",
            "feature id",
            "#otu id",
            "otu id",
            "index",
        }

        for column in dataframe.columns:
            normalized = (
                str(column)
                .strip()
                .lower()
            )

            if normalized in aliases:
                return column

        raise ValueError(
            "Could not identify the feature-ID column.\n"
            f"Available columns: {list(dataframe.columns)}"
        )


    def find_taxon_column(
        dataframe: pd.DataFrame,
    ) -> str:
        """
        Detect the taxonomy-label column.
        """
        aliases = {
            "taxon",
            "taxonomy",
        }

        for column in dataframe.columns:
            normalized = (
                str(column)
                .strip()
                .lower()
            )

            if normalized in aliases:
                return column

        raise ValueError(
            "Could not identify the taxon column.\n"
            f"Available columns: {list(dataframe.columns)}"
        )


    def load_raw_depths(
        table_path: Path,
        run_mapping: dict[str, str],
        metadata_ids: list[str],
    ) -> pd.Series:
        """
        Calculate original sample read depths directly from
        the unfiltered exported BIOM-to-TSV table.
        """
        table = read_biom_tsv(table_path)

        feature_id_column = table.columns[0]
        sample_columns = list(table.columns[1:])

        numeric_counts = (
            table[sample_columns]
            .apply(pd.to_numeric, errors="raise")
        )

        raw_depths = numeric_counts.sum(axis=0)

        normalized_ids = [
            run_mapping.get(
                str(sample_id).strip(),
                str(sample_id).strip(),
            )
            for sample_id in raw_depths.index
        ]

        if len(set(normalized_ids)) != len(normalized_ids):
            raise ValueError(
                f"Duplicate sample IDs after mapping in {table_path}"
            )

        raw_depths.index = normalized_ids

        missing_ids = (
            set(metadata_ids) - set(raw_depths.index)
        )

        if missing_ids:
            raise ValueError(
                f"Raw table is missing metadata samples: "
                f"{sorted(missing_ids)}"
            )

        return raw_depths.loc[metadata_ids]


    metadata = pd.read_csv(
        METADATA_FILE,
        sep="\t",
    )

    metadata_ids = (
        metadata[MODEL_ID_COLUMN]
        .astype(str)
        .str.strip()
        .tolist()
    )

    expected_sample_count = CONFIG['ml'].get('expected_sample_count')
    if expected_sample_count is not None and len(metadata_ids) != int(expected_sample_count):
        raise ValueError(f"Expected {expected_sample_count} metadata samples, found {len(metadata_ids)}")

    if len(set(metadata_ids)) != len(metadata_ids):
        raise ValueError(
            "Duplicate participant IDs found in metadata."
        )


    mapping_file = (
        MATRIX_ROOT
        / "run_to_participant_mapping.tsv"
    )

    mapping_table = pd.read_csv(
        mapping_file,
        sep="\t",
    )

    run_id_column = CONFIG['input']['run_id_column']
    required_mapping_columns = {run_id_column, MODEL_ID_COLUMN}

    missing_mapping_columns = (
        required_mapping_columns
        - set(mapping_table.columns)
    )

    if missing_mapping_columns:
        raise ValueError(
            f"Mapping file is missing columns: "
            f"{sorted(missing_mapping_columns)}"
        )

    run_to_participant = dict(
        zip(
            mapping_table[run_id_column]
            .astype(str)
            .str.strip(),
            mapping_table[MODEL_ID_COLUMN]
            .astype(str)
            .str.strip(),
        )
    )

    qc_rows = []

    for branch in [BRANCH_KEY]:

        branch_dir = MATRIX_ROOT / branch

        counts = load_matrix(
            branch_dir / "counts.tsv"
        )

        relative_abundance = load_matrix(
            branch_dir / "relative_abundance.tsv"
        )

        clr = load_matrix(
            branch_dir / "clr.tsv"
        )

        feature_metadata = pd.read_csv(
            branch_dir / "feature_metadata.tsv",
            sep="\t",
        )

        print(
            f"\nChecking {branch} branch..."
        )

        print(
            "Feature metadata columns:",
            list(feature_metadata.columns),
        )

        # --------------------------------------------------------
        # Sample alignment
        # --------------------------------------------------------

        if list(counts.index) != metadata_ids:
            raise ValueError(
                f"Sample order mismatch in {branch} counts matrix."
            )

        if list(relative_abundance.index) != metadata_ids:
            raise ValueError(
                f"Sample order mismatch in {branch} "
                "relative-abundance matrix."
            )

        if list(clr.index) != metadata_ids:
            raise ValueError(
                f"Sample order mismatch in {branch} CLR matrix."
            )

        # --------------------------------------------------------
        # Feature alignment
        # --------------------------------------------------------

        if list(counts.columns) != list(
            relative_abundance.columns
        ):
            raise ValueError(
                f"Feature mismatch between counts and "
                f"relative abundance for {branch}."
            )

        if list(counts.columns) != list(
            clr.columns
        ):
            raise ValueError(
                f"Feature mismatch between counts and "
                f"CLR for {branch}."
            )

        feature_id_column = find_feature_id_column(
            feature_metadata
        )

        taxon_column = find_taxon_column(
            feature_metadata
        )

        feature_ids = (
            feature_metadata[feature_id_column]
            .astype(str)
            .str.strip()
        )

        matrix_feature_ids = (
            pd.Series(counts.columns)
            .astype(str)
            .str.strip()
        )

        if set(feature_ids) != set(matrix_feature_ids):
            missing_from_metadata = (
                set(matrix_feature_ids) - set(feature_ids)
            )

            missing_from_matrix = (
                set(feature_ids) - set(matrix_feature_ids)
            )

            raise ValueError(
                f"Feature metadata mismatch for {branch}\n"
                f"Feature-ID column used: {feature_id_column}\n"
                f"Missing from feature metadata: "
                f"{sorted(missing_from_metadata)}\n"
                f"Missing from matrix: "
                f"{sorted(missing_from_matrix)}"
            )

        # --------------------------------------------------------
        # Relative-abundance validation
        # --------------------------------------------------------

        relative_row_sums = (
            relative_abundance.sum(axis=1)
        )

        if not np.allclose(
            relative_row_sums.to_numpy(),
            1.0,
            atol=1e-10,
        ):
            raise ValueError(
                f"Relative-abundance rows do not sum to 1 "
                f"for {branch}."
            )

        # --------------------------------------------------------
        # CLR validation
        # --------------------------------------------------------

        clr_row_means = clr.mean(axis=1)

        if not np.allclose(
            clr_row_means.to_numpy(),
            0.0,
            atol=1e-10,
        ):
            raise ValueError(
                f"CLR rows are not centered at zero "
                f"for {branch}."
            )

        # --------------------------------------------------------
        # Taxonomy validation
        # --------------------------------------------------------

        unassigned_count = (
            feature_metadata[taxon_column]
            .fillna("Unassigned")
            .astype(str)
            .str.strip()
            .eq("Unassigned")
            .sum()
        )

        qc_rows.append(
            {
                "branch": branch,
                "sample_count": len(counts),
                "feature_count": len(counts.columns),
                "feature_id_column": feature_id_column,
                "taxon_column": taxon_column,
                "unassigned_taxa": int(unassigned_count),
                "relative_abundance_valid": True,
                "clr_valid": True,
                "min_filtered_sample_total": float(
                    counts.sum(axis=1).min()
                ),
                "median_filtered_sample_total": float(
                    counts.sum(axis=1).median()
                ),
                "max_filtered_sample_total": float(
                    counts.sum(axis=1).max()
                ),
            }
        )

        print(
            f"{branch}: sample alignment passed"
        )

        print(
            f"{branch}: feature alignment passed"
        )

        print(
            f"{branch}: relative-abundance validation passed"
        )

        print(
            f"{branch}: CLR validation passed"
        )

    # ------------------------------------------------------------
    # Compare original, pre-filtering sample depths
    # ------------------------------------------------------------

    # Check original pre-filtering depths for the selected branch only

    selected_export_dir = (
        "asv_table"
        if BRANCH_KEY == "asv"
        else "otu_table"
    )

    selected_raw_table = (
        EXPORT_ROOT
        / selected_export_dir
        / "feature-table.tsv"
    )

    if not selected_raw_table.is_file():
        raise FileNotFoundError(
            f"Selected-branch raw table not found:\n"
            f"{selected_raw_table}\n"
            f"Selected branch: {BRANCH_KEY}"
        )

    selected_raw_depths = load_raw_depths(
        selected_raw_table,
        run_to_participant,
        metadata_ids,
    )

    raw_depth_summary = {
        "branch": BRANCH_KEY,
        "raw_sample_count": len(selected_raw_depths),
        "raw_min_sample_depth": int(
            selected_raw_depths.min()
        ),
        "raw_median_sample_depth": float(
            selected_raw_depths.median()
        ),
        "raw_max_sample_depth": int(
            selected_raw_depths.max()
        ),
        "raw_total_reads": int(
            selected_raw_depths.sum()
        ),
    }

    qc_summary = pd.DataFrame(qc_rows)

    qc_summary.to_csv(
        MATRIX_ROOT / "matrix_qc_summary.tsv",
        sep="\t",
        index=False,
    )

    print("\nSelected-branch matrix QC passed.")
    print(qc_summary.to_string(index=False))

    print("\nOriginal raw read-depth summary:")
    for key, value in raw_depth_summary.items():
        print(f"{key}: {value}")

    print(
        f"\nRaw-depth check completed for selected branch: "
        f"{BRANCH_KEY}"
    )



Checking asv branch...
Feature metadata columns: ['#OTU ID', 'taxon', 'prevalence_samples', 'total_count', 'branch']
asv: sample alignment passed
asv: feature alignment passed
asv: relative-abundance validation passed
asv: CLR validation passed

Selected-branch matrix QC passed.
branch  sample_count  feature_count feature_id_column taxon_column  unassigned_taxa  relative_abundance_valid  clr_valid  min_filtered_sample_total  median_filtered_sample_total  max_filtered_sample_total
   asv            88            511           #OTU ID        taxon                0                      True       True                    15029.0                       28057.5                    71194.0

Original raw read-depth summary:
branch: asv
raw_sample_count: 88
raw_min_sample_depth: 17900
raw_median_sample_depth: 40181.0
raw_max_sample_depth: 84036
raw_total_reads: 3591410

Raw-depth check completed for selected branch: asv


In [ ]:
# Section 5 preparation: obtain the upstream ML implementation used below.
if not ML_ENABLED:
    print("ML disabled: skipping bundled ML snapshot validation.")
else:
    required_ml_scripts = [
        ML_REPO / 'dataset1_filtered_diet_ml_pipeline.py',
        ML_REPO / 'dataset1_svm__plot.R',
    ]
    missing_ml_scripts = [path for path in required_ml_scripts if not path.is_file()]
    if missing_ml_scripts:
        raise FileNotFoundError('Bundled ML snapshot is incomplete: ' + ', '.join(map(str, missing_ml_scripts)))
    print('Using bundled ML snapshot:', ML_REPO)


## 5. Leakage-controlled ML evaluation


In [ ]:
# Fit and evaluate the configured models on the selected CLR matrix.

if not ML_ENABLED:
    print("ML disabled for this execution mode; skipping machine learning.")
else:
    # run ML on the selected CLR matrix

    from pathlib import Path
    import importlib.util
    import inspect

    import numpy as np
    import pandas as pd

    from sklearn.model_selection import StratifiedShuffleSplit
    from sklearn.preprocessing import StandardScaler


    # ------------------------------------------------------------
    # Paths
    # ------------------------------------------------------------


    WEEK4_ROOT=Path(CONFIG['run']['ml_dir'])
    MATRIX_ROOT=WEEK4_ROOT/'matrices'
    BRANCH_KEY='otu' if CONFIG['run']['branch']=='otu' else 'asv'
    # Keep ML-only reruns compatible with matrices created by the older otu_97 naming.
    if BRANCH_KEY == 'otu':
        canonical_matrix_dir = MATRIX_ROOT / 'otu'
        legacy_matrix_dir = MATRIX_ROOT / 'otu_97'
        if not canonical_matrix_dir.exists() and legacy_matrix_dir.exists():
            shutil.copytree(legacy_matrix_dir, canonical_matrix_dir)
    required_ml_inputs = [
        PHENOTYPE_FILE,
        MATRIX_ROOT / BRANCH_KEY / 'clr.tsv',
        MATRIX_ROOT / BRANCH_KEY / 'feature_metadata.tsv',
    ]
    missing_ml_inputs = [path for path in required_ml_inputs if not path.is_file()]
    if missing_ml_inputs:
        raise FileNotFoundError(
            'ML mode requires existing matrix inputs; missing: '
            + ', '.join(map(str, missing_ml_inputs))
        )
    OUTPUT_ROOT=WEEK4_ROOT/f'qiime_ml_results_{BRANCH_KEY}'

    OUTPUT_ROOT.mkdir(
        parents=True,
        exist_ok=True,
    )


    # ------------------------------------------------------------
    # Load the repository's ML functions
    # ------------------------------------------------------------

    repo_script = (
        ML_REPO
        / "dataset1_filtered_diet_ml_pipeline.py"
    )

    spec = importlib.util.spec_from_file_location(
        "repository_ml_pipeline",
        repo_script,
    )

    repository_ml = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(repository_ml)

    selector_source = inspect.getsource(repository_ml.select_taxa)
    if "Boruta" in selector_source or "boruta" in selector_source:
        raise RuntimeError(
            "The plotting integration requires the clean three-method GitHub selector; Boruta was detected."
        )


    RANDOM_STATES=tuple(int(x) for x in CONFIG['ml']['random_states'])


    # ------------------------------------------------------------
    # Helper functions
    # ------------------------------------------------------------

    def load_qiime_branch(
        branch_name: str,
        metadata: pd.DataFrame,
    ):
        branch_dir = MATRIX_ROOT / branch_name

        clr = pd.read_csv(
            branch_dir / "clr.tsv",
            sep="\t",
        )

        feature_metadata = pd.read_csv(
            branch_dir / "feature_metadata.tsv",
            sep="\t",
        )

        if MODEL_ID_COLUMN not in clr.columns:
            raise ValueError(
                f"{MODEL_ID_COLUMN} is missing from {branch_name} CLR table."
            )

        clr = clr.set_index(MODEL_ID_COLUMN)

        metadata = metadata.copy()
        metadata[MODEL_ID_COLUMN] = (
            metadata[MODEL_ID_COLUMN]
            .astype(str)
            .str.strip()
        )

        clr.index = (
            clr.index
            .astype(str)
            .str.strip()
        )

        metadata_ids = set(
            metadata[MODEL_ID_COLUMN]
        )

        clr_ids = set(clr.index)

        if metadata_ids != clr_ids:
            raise ValueError(
                f"Sample mismatch in {branch_name} CLR matrix.\n"
                f"Missing from metadata: "
                f"{sorted(clr_ids - metadata_ids)}\n"
                f"Missing from CLR matrix: "
                f"{sorted(metadata_ids - clr_ids)}"
            )

        metadata_aligned = (
            metadata
            .set_index(MODEL_ID_COLUMN)
            .loc[clr.index]
            .copy()
        )

        if metadata_aligned[OUTCOME_COLUMN].isna().any():
            raise ValueError(
                f"Missing depression labels in {branch_name}."
            )

        negative_class = str(CONFIG['ml']['negative_class'])
        positive_class = str(CONFIG['ml']['positive_class'])
        labels = metadata_aligned[OUTCOME_COLUMN].astype(str).str.strip()
        if set(labels.unique()) != {negative_class, positive_class}:
            raise ValueError(f'Outcome labels are not the configured binary classes in {branch_name}.')
        y = np.where(labels.eq(positive_class), 1, 0)

        missing_diet_columns = (
            set(DIET_COLUMNS)
            - set(metadata_aligned.columns)
        )

        if missing_diet_columns:
            raise ValueError(
                f"Missing diet columns: "
                f"{sorted(missing_diet_columns)}"
            )

        x_taxa = clr.to_numpy(dtype=float)

        if not np.isfinite(x_taxa).all():
            raise ValueError(
                f"Non-finite CLR values found in {branch_name}."
            )

        taxa_ids = clr.columns.astype(str).tolist()

        # Detect the feature-ID column in the taxonomy metadata.
        feature_id_aliases = {
            "feature_id",
            "feature id",
            "#otu id",
            "otu id",
            "index",
        }

        feature_id_column = None

        for column in feature_metadata.columns:
            normalized = (
                str(column)
                .strip()
                .lower()
            )

            if normalized in feature_id_aliases:
                feature_id_column = column
                break

        if feature_id_column is None:
            raise ValueError(
                f"Could not identify the feature-ID column for "
                f"{branch_name}. Available columns: "
                f"{list(feature_metadata.columns)}"
            )

        if "taxon" not in feature_metadata.columns:
            raise ValueError(
                f"Taxon column is missing for {branch_name}."
            )

        feature_metadata[feature_id_column] = (
            feature_metadata[feature_id_column]
            .astype(str)
            .str.strip()
        )

        taxon_lookup = (
            feature_metadata
            .drop_duplicates(feature_id_column)
            .set_index(feature_id_column)["taxon"]
            .fillna("Unassigned")
            .astype(str)
            .to_dict()
        )

        missing_taxa = [
            feature_id
            for feature_id in taxa_ids
            if feature_id not in taxon_lookup
        ]

        if missing_taxa:
            raise ValueError(
                f"Some CLR features lack taxonomy metadata in "
                f"{branch_name}: {missing_taxa[:10]}"
            )

        return (
            metadata_aligned,
            x_taxa,
            y,
            taxa_ids,
            taxon_lookup,
        )


    def run_qiime_branch(
        branch_name: str,
        metadata: pd.DataFrame,
    ):
        (
            metadata_aligned,
            x_all_taxa,
            y,
            taxa_ids,
            taxon_lookup,
        ) = load_qiime_branch(
            branch_name,
            metadata,
        )

        x_diet = (
            metadata_aligned
            .loc[:, DIET_COLUMNS]
            .to_numpy(dtype=float)
        )

        result_rows = []
        selection_rows = []
        svm_prediction_rows = []
        svm_coefficient_rows = []

        print(
            f"\nRunning branch: {branch_name}"
        )

        print(
            f"Samples: {len(y)} | "
            f"Controls: {(y == 0).sum()} | "
            f"Depressive: {(y == 1).sum()}"
        )

        print(
            f"Candidate microbiome features: "
            f"{x_all_taxa.shape[1]}"
        )

        for split_number, random_state in enumerate(
            RANDOM_STATES,
            start=1,
        ):

            # First split:
            # 40% feature selection, 60% train/test pool.
            first_split = StratifiedShuffleSplit(
                n_splits=1,
                test_size=float(CONFIG['ml']['feature_selection_percent']) / 100.0,
                random_state=random_state,
            )

            train_test_indices, fs_indices = next(
                first_split.split(
                    x_all_taxa,
                    y,
                )
            )

            # Second split:
            # approximately 40% training and 20% testing.
            second_split = StratifiedShuffleSplit(
                n_splits=1,
                test_size=float(CONFIG['ml']['test_percent']) / (float(CONFIG['ml']['train_percent']) + float(CONFIG['ml']['test_percent'])),
                random_state=random_state,
            )

            train_relative, test_relative = next(
                second_split.split(
                    x_all_taxa[train_test_indices],
                    y[train_test_indices],
                )
            )

            train_indices = (
                train_test_indices[train_relative]
            )

            test_indices = (
                train_test_indices[test_relative]
            )

            # Feature selection uses only the feature-selection subset.
            selected_taxa_indices, method_sets, mi_threshold, rf_threshold = (
                repository_ml.select_taxa(
                    x_all_taxa[fs_indices],
                    y[fs_indices],
                    random_state,
                )
            )

            selected_taxa_ids = [
                taxa_ids[index]
                for index in selected_taxa_indices
            ]

            for feature_index, feature_id in enumerate(
                taxa_ids
            ):
                selection_rows.append(
                    {
                        "branch": branch_name,
                        "Split": split_number,
                        "random_state": random_state,
                        "Feature ID": feature_id,
                        "Taxon": taxon_lookup[feature_id],
                        "ANOVA": (
                            feature_index
                            in method_sets["ANOVA"]
                        ),
                        "MI": (
                            feature_index
                            in method_sets["MI"]
                        ),
                        "RF": (
                            feature_index
                            in method_sets["RF"]
                        ),
                        "Consensus": (
                            feature_index
                            in selected_taxa_indices
                        ),
                        "MI threshold": mi_threshold,
                        "RF threshold": rf_threshold,
                    }
                )

            x_selected_taxa = (
                x_all_taxa[:, selected_taxa_indices]
            )

            diet_ids = list(DIET_COLUMNS)
            diet_names = [
                getattr(
                    repository_ml,
                    "DIET_NAME_MAP",
                    {
                        "Prot": "Protein",
                        "SatFat": "Saturated fat",
                        "Poly_InsFat": "Polyunsaturated fat",
                        "Chol": "Cholesterol",
                        "Carb": "Carbohydrate",
                        "Fiber": "Fibre",
                    },
                ).get(feature_id, feature_id)
                for feature_id in diet_ids
            ]
            diet_groups = ["Diet"] * len(diet_ids)
            taxa_names = [
                taxon_lookup[feature_id]
                for feature_id in selected_taxa_ids
            ]
            taxa_groups = ["Taxa"] * len(selected_taxa_ids)

            feature_sets = {
                "Diet": {
                    "matrix": x_diet,
                    "ids": diet_ids,
                    "names": diet_names,
                    "groups": diet_groups,
                },
                "Taxa": {
                    "matrix": x_selected_taxa,
                    "ids": selected_taxa_ids,
                    "names": taxa_names,
                    "groups": taxa_groups,
                },
                "Diet + taxa": {
                    "matrix": np.hstack([x_diet, x_selected_taxa]),
                    "ids": diet_ids + selected_taxa_ids,
                    "names": diet_names + taxa_names,
                    "groups": diet_groups + taxa_groups,
                },
            }

            for input_name, feature_data in feature_sets.items():
                x_features = feature_data["matrix"]

                classifiers = (
                    repository_ml.build_classifiers()
                )

                for model_name, model in classifiers.items():

                    scaler = StandardScaler()

                    x_train = scaler.fit_transform(
                        x_features[train_indices]
                    )

                    x_test = scaler.transform(
                        x_features[test_indices]
                    )

                    y_train = y[train_indices]
                    y_test = y[test_indices]

                    model.fit(
                        x_train,
                        y_train,
                    )

                    predictions = model.predict(
                        x_test
                    )

                    scores = (
                        repository_ml.continuous_scores(
                            model,
                            x_test,
                        )
                    )

                    metrics = (
                        repository_ml.calculate_metrics(
                            y_test,
                            predictions,
                            scores,
                        )
                    )

                    result_rows.append(
                        {
                            "Branch": branch_name,
                            "Split": split_number,
                            "random_state": random_state,
                            "Input features": input_name,
                            "Model": model_name,
                            "n candidate taxa": len(taxa_ids),
                            "n selected taxa": len(
                                selected_taxa_indices
                            ),
                            "n diet features": len(
                                DIET_COLUMNS
                            ),
                            "n total features": (
                                x_features.shape[1]
                            ),
                            **metrics,
                        }
                    )

                    if model_name == "SVM" and input_name == "Diet + taxa":
                        for y_value, prediction, score in zip(
                            y_test, predictions, scores
                        ):
                            svm_prediction_rows.append(
                                {
                                    "Branch": branch_name,
                                    "Split": split_number,
                                    "random_state": random_state,
                                    "y_true": int(y_value),
                                    "y_pred": int(prediction),
                                    "score": float(score),
                                }
                            )

                        coefficients = np.asarray(
                            model.coef_[0],
                            dtype=float,
                        )
                        absolute_coefficients = np.abs(coefficients)
                        order = np.argsort(-absolute_coefficients)
                        ranks = np.empty_like(order)
                        ranks[order] = np.arange(len(order)) + 1

                        for feature_index, (
                            feature_id,
                            feature_name,
                            feature_group,
                        ) in enumerate(
                            zip(
                                feature_data["ids"],
                                feature_data["names"],
                                feature_data["groups"],
                            )
                        ):
                            svm_coefficient_rows.append(
                                {
                                    "Branch": branch_name,
                                    "Split": split_number,
                                    "random_state": random_state,
                                    "Feature ID": feature_id,
                                    "Feature name": feature_name,
                                    "Feature group": feature_group,
                                    "Coefficient": float(coefficients[feature_index]),
                                    "Absolute coefficient": float(absolute_coefficients[feature_index]),
                                    "Rank by absolute coefficient": int(ranks[feature_index]),
                                    "Ranked in top 20": bool(ranks[feature_index] <= 20),
                                }
                            )

            print(
                f"Split {split_number}/{len(RANDOM_STATES)} completed "
                f"with {len(selected_taxa_indices)} "
                "selected taxa."
            )

        output_dir = OUTPUT_ROOT / branch_name
        output_dir.mkdir(
            parents=True,
            exist_ok=True,
        )

        classification_results = pd.DataFrame(
            result_rows
        )

        selection_results = pd.DataFrame(
            selection_rows
        )

        svm_predictions = pd.DataFrame(
            svm_prediction_rows
        )

        svm_coefficients = pd.DataFrame(
            svm_coefficient_rows
        )

        performance_summary = (
            classification_results
            .groupby(
                [
                    "Branch",
                    "Input features",
                    "Model",
                ],
                as_index=False,
            )
            [
                [
                    "ROC-AUC",
                    "ACC",
                    "BAL-ACC",
                    "SEN",
                    "SPE",
                    "PRE",
                    "F1",
                ]
            ]
            .mean()
        )

        taxa_selection_frequency = (
            selection_results
            .groupby(
                [
                    "Feature ID",
                    "Taxon",
                ],
                as_index=False,
            )
            .agg(
                Splits_selected_by_consensus=(
                    "Consensus",
                    "sum",
                ),
                Splits_selected_by_ANOVA=(
                    "ANOVA",
                    "sum",
                ),
                Splits_selected_by_MI=(
                    "MI",
                    "sum",
                ),
                Splits_selected_by_RF=(
                    "RF",
                    "sum",
                ),
            )
            .sort_values(
                [
                    "Splits_selected_by_consensus",
                    "Taxon",
                ],
                ascending=[
                    False,
                    True,
                ],
            )
        )

        classification_results.to_csv(
            output_dir
            / "classification_results_all_splits.tsv",
            sep="\t",
            index=False,
        )

        classification_results.to_csv(
            output_dir / "classification_results_all_splits.csv",
            index=False,
        )

        svm_predictions.to_csv(
            output_dir / "svm_predictions_all_splits.csv",
            index=False,
        )

        svm_coefficients.to_csv(
            output_dir / "svm_coefficients_all_splits.csv",
            index=False,
        )

        performance_summary.to_csv(
            output_dir
            / "performance_summary.tsv",
            sep="\t",
            index=False,
        )

        selection_results.to_csv(
            output_dir
            / "taxa_feature_selection_all_splits.tsv",
            sep="\t",
            index=False,
        )

        taxa_selection_frequency.to_csv(
            output_dir
            / "taxa_selection_frequency.tsv",
            sep="\t",
            index=False,
        )

        return (
            classification_results,
            performance_summary,
            taxa_selection_frequency,
        )


    metadata=pd.read_csv(PHENOTYPE_FILE,sep='\t')
    branch_results=run_qiime_branch(BRANCH_KEY,metadata)
    performance_summary=branch_results[1]
    performance_summary.to_csv(OUTPUT_ROOT/'performance_summary.tsv',sep='\t',index=False)
    print('Selected ML branch completed:',BRANCH_KEY)
    print(performance_summary.to_string(index=False))



Running branch: asv
Samples: 88 | Controls: 73 | Depressive: 15
Candidate microbiome features: 511
Split 1/5 completed with 39 selected taxa.
Split 2/5 completed with 25 selected taxa.
Split 3/5 completed with 34 selected taxa.
Split 4/5 completed with 45 selected taxa.
Split 5/5 completed with 38 selected taxa.
Selected ML branch completed: asv
Branch Input features Model  ROC-AUC      ACC  BAL-ACC      SEN      SPE      PRE       F1
   asv           Diet   BRF 0.440000 0.622222 0.480000 0.266667 0.693333 0.157143 0.191111
   asv           Diet    LR 0.551111 0.555556 0.546667 0.533333 0.560000 0.214408 0.288333
   asv           Diet    RF 0.413333 0.777778 0.466667 0.000000 0.933333 0.000000 0.000000
   asv           Diet   SVM 0.600000 0.600000 0.600000 0.600000 0.600000 0.244963 0.330758
   asv    Diet + taxa   BRF 0.588889 0.722222 0.646667 0.533333 0.760000 0.341905 0.413016
   asv    Diet + taxa    LR 0.395556 0.588889 0.406667 0.133333 0.680000 0.073333 0.094444
   asv    Diet

### 5.1 Repeated-split SVM figures


In [ ]:
%%bash
set -euo pipefail
if [ "${ML_ENABLED:-true}" != "true" ]; then
  echo "ML disabled: skipping ML plotting dependencies."
  exit 0
fi
if ! command -v Rscript >/dev/null 2>&1; then
  apt-get update -qq
  apt-get install -y -qq r-base r-base-dev
fi
Rscript -e "required <- c('dplyr','ggplot2','pROC','patchwork'); missing <- required[!vapply(required, requireNamespace, logical(1), quietly=TRUE)]; if (length(missing)) install.packages(missing, repos='https://cloud.r-project.org')"


g++ -std=gnu++20 -I"/usr/share/R/include" -DNDEBUG  -I'/usr/lib/R/site-library/Rcpp/include'     -fpic  -g -O2 -ffile-prefix-map=/build/r-base-QML2qB/r-base-4.6.1=. -fstack-protector-strong -Wformat -Werror=format-security -Wdate-time -D_FORTIFY_SOURCE=2   -c RcppExports.cpp -o RcppExports.o
g++ -std=gnu++20 -I"/usr/share/R/include" -DNDEBUG  -I'/usr/lib/R/site-library/Rcpp/include'     -fpic  -g -O2 -ffile-prefix-map=/build/r-base-QML2qB/r-base-4.6.1=. -fstack-protector-strong -Wformat -Werror=format-security -Wdate-time -D_FORTIFY_SOURCE=2   -c RcppVersion.cpp -o RcppVersion.o
g++ -std=gnu++20 -I"/usr/share/R/include" -DNDEBUG  -I'/usr/lib/R/site-library/Rcpp/include'     -fpic  -g -O2 -ffile-prefix-map=/build/r-base-QML2qB/r-base-4.6.1=. -fstack-protector-strong -Wformat -Werror=format-security -Wdate-time -D_FORTIFY_SOURCE=2   -c delong.cpp -o delong.o
g++ -std=gnu++20 -shared -L/usr/lib/R/lib -Wl,-Bsymbolic-functions -flto=auto -ffat-lto-objects -flto=auto -Wl,-z,relro -o pROC.so 

Installing packages into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)
trying URL 'https://cloud.r-project.org/src/contrib/pROC_1.19.0.1.tar.gz'
trying URL 'https://cloud.r-project.org/src/contrib/patchwork_1.3.2.tar.gz'
* installing *source* package ‘pROC’ ...
** this is package ‘pROC’ version ‘1.19.0.1’
** package ‘pROC’ successfully unpacked and MD5 sums checked
** using staged installation
** libs
using C++ compiler: ‘g++ (Ubuntu 11.4.0-1ubuntu1~22.04.3) 11.4.0’
installing to /usr/local/lib/R/site-library/00LOCK-pROC/00new/pROC/libs
** R
** data
*** moving datasets to lazyload DB
** inst
** byte-compile and prepare package for lazy loading
** help
*** installing help indices
** building package indices
** testing if installed package can be loaded from temporary location
** checking absolute paths in shared objects and dynamic libraries
** testing if installed package can be loaded from final location
** testing if installed package keeps a record of temporary installat

In [ ]:
# Generate ML figures from the completed result tables.
if not ML_ENABLED:
    print("ML disabled: skipping plot generation.")
else:
    from pathlib import Path
    import subprocess

    PLOT_RESULTS = OUTPUT_ROOT / BRANCH_KEY
    PLOT_DIR = PLOT_RESULTS / "plots"
    PLOT_DIR.mkdir(parents=True, exist_ok=True)
    PLOT_SCRIPT = ML_REPO / "dataset1_svm__plot.R"

    if not PLOT_SCRIPT.is_file():
        raise FileNotFoundError(f"Plot script not found: {PLOT_SCRIPT}")

    subprocess.run(
        ["Rscript", str(PLOT_SCRIPT), str(PLOT_RESULTS), str(PLOT_DIR)],
        check=True,
    )

    print("Plot outputs:")
    for path in sorted(PLOT_DIR.iterdir()):
        print(path)


Plot outputs:
/content/week4_dataset1/qiime_ml_results_asv/asv/plots/01_Model_AUC_Across_Five_Repeated_Splits.pdf
/content/week4_dataset1/qiime_ml_results_asv/asv/plots/01_Model_AUC_Across_Five_Repeated_Splits.png
/content/week4_dataset1/qiime_ml_results_asv/asv/plots/02_Mean_ROC_Curve_SVM_Model.pdf
/content/week4_dataset1/qiime_ml_results_asv/asv/plots/02_Mean_ROC_Curve_SVM_Model.png
/content/week4_dataset1/qiime_ml_results_asv/asv/plots/03_Pooled_Confusion_Matrix_SVM_Model.pdf
/content/week4_dataset1/qiime_ml_results_asv/asv/plots/03_Pooled_Confusion_Matrix_SVM_Model.png
/content/week4_dataset1/qiime_ml_results_asv/asv/plots/04_SVM_Coefficients_Across_Five_Repeated_Splits.pdf
/content/week4_dataset1/qiime_ml_results_asv/asv/plots/04_SVM_Coefficients_Across_Five_Repeated_Splits.png


In [ ]:
# Assemble a compact workbook from the validated ML outputs.
if not ML_ENABLED:
    print("ML disabled: skipping the ML workbook.")
else:
    from openpyxl import load_workbook
    from openpyxl.styles import Font, PatternFill
    WORKBOOK=ML_ROOT/f'{CONFIG["run"]["name"]}_{BRANCH_KEY}_results.xlsx'
    with pd.ExcelWriter(WORKBOOK,engine='openpyxl') as writer:
        pd.DataFrame([['Branch',BRANCH_KEY],['Truncation F/R',f'{CONFIG["quality"]["trunc_len_f"]}/{CONFIG["quality"]["trunc_len_r"]}'],['Sampling depth',CONFIG['quality']['sampling_depth']],['Primer F',CONFIG['primers']['forward']],['Primer R',CONFIG['primers']['reverse']]],columns=['Parameter','Value']).to_excel(writer,sheet_name='README',index=False)
        pd.read_csv(ML_ROOT/'matrices/matrix_build_summary.tsv',sep='\t').to_excel(writer,sheet_name='Matrix_summary',index=False)
        pd.read_csv(ML_ROOT/'matrices/matrix_qc_summary.tsv',sep='\t').to_excel(writer,sheet_name='Matrix_QC',index=False)
        pd.read_csv(ML_ROOT/f'qiime_ml_results_{BRANCH_KEY}/performance_summary.tsv',sep='\t').to_excel(writer,sheet_name='ML_performance',index=False)
        pd.read_csv(ML_ROOT/f'matrices/{BRANCH_KEY}/feature_metadata.tsv',sep='\t').to_excel(writer,sheet_name='Feature_metadata',index=False)
    wb=load_workbook(WORKBOOK)
    for ws in wb.worksheets:
        ws.freeze_panes='A2'
        for cell in ws[1]: cell.font=Font(bold=True,color='FFFFFF'); cell.fill=PatternFill('solid',fgColor='4F81BD')
    wb.save(WORKBOOK)
    print('Workbook:',WORKBOOK)


Workbook: /content/week4_dataset1/colombian_16s_asv_results.xlsx


## 6. Archive and provenance


In [ ]:
# Archive the completed outputs so the next rerun can restore DADA2 instead of recomputing it.
archive_dir = Path(CONFIG['run']['archive_dir'])
archive_dir.mkdir(parents=True, exist_ok=True)
archive = archive_dir / f"{CONFIG['run']['name']}_{BRANCH_KEY}_results.tar.gz"
manifest = archive_dir / f"{CONFIG['run']['name']}_{BRANCH_KEY}_manifest.tsv"
checksum = archive_dir / f"{CONFIG['run']['name']}_{BRANCH_KEY}.sha256"

if CONFIG['reporting'].get('archive_outputs', True):
    def add_if_exists(tar, path, arcname):
        path = Path(path)
        if path.exists():
            tar.add(path, arcname=arcname)

    with tarfile.open(archive, 'w:gz') as tf:
        if BIOLOGY_ENABLED:
            add_if_exists(tf, RESULTS_ROOT, 'results/colombian_full')
        if ML_ENABLED:
            add_if_exists(tf, ML_ROOT / 'matrices', 'ml/matrices')
            add_if_exists(tf, ML_ROOT / f'qiime_ml_results_{BRANCH_KEY}', f'ml/qiime_ml_results_{BRANCH_KEY}')
            if 'WORKBOOK' in globals():
                add_if_exists(tf, WORKBOOK, Path(WORKBOOK).name)
        add_if_exists(tf, PREFERENCES_FILE, 'preferences.yaml')
        if 'QIIME_METADATA' in globals():
            add_if_exists(tf, QIIME_METADATA, 'metadata.tsv')
        if 'PHENOTYPE_FILE' in globals():
            add_if_exists(tf, PHENOTYPE_FILE, Path(PHENOTYPE_FILE).name)
    with tarfile.open(archive, 'r:gz') as tf:
        rows = [f'{member.size}\t{member.name}' for member in sorted(tf.getmembers(), key=lambda item: item.name) if member.isfile()]
    manifest.write_text('size_bytes\tpath\n' + '\n'.join(rows) + '\n')
    checksum.write_text(hashlib.sha256(archive.read_bytes()).hexdigest() + '  ' + archive.name + '\n')
    print(archive); print(manifest); print(checksum)


In [ ]:
print('Pipeline completed:', BRANCH, '| biology:', BIOLOGY_ENABLED, '| ML:', ML_ENABLED)
if BIOLOGY_ENABLED:
    print('DADA2 source:', 'Drive archive' if RESTORED_FROM_ARCHIVE else 'Nextflow')
    print('DADA2/branch results:', RESULTS_ROOT)
if ML_ENABLED and 'WORKBOOK' in globals():
    print('Workbook:', WORKBOOK)
